# 4.6 Your turn: distributions in your own chat

[04.1](04.1-parametric-distributions.ipynb) through [04.5](04.5-fitting-the-residual.ipynb)
built the six-family toolkit — the registry, `DistributionFitter`, `fit_table`'s two
winners, `PlotFits`, `QQPlot`, `ECDFPlot` — on data chosen so a shape was guaranteed to be
there. This notebook makes the same move [03.3](../lesson3/roos_03.3-events-in-your-chat.ipynb)
and [02.5](../lesson2/roos_02.5-your_turn.ipynb) already made: your data, your question,
and nothing guaranteed.

**x-axis: the value. y-axis: how likely it is** — a density for something continuous
(seconds, characters), a probability mass for something discrete (a count). *Which*
variable, and *which* family, is what the rest of this notebook works out with you rather
than for you.

In [ ]:
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize, stats

from goad_toolkit.analytics import DistributionFitter, FitResult, NullDistribution, fit_table
from goad_toolkit.datatransforms import CountValues, Pipeline, RegexFeature, TimeFeatures, TransformBase
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import (
    DistPlot,
    ECDFPlot,
    FitPlotSettings,
    HistogramPlot,
    HorizontalLine,
    LinePlot,
    NullPlot,
    PlotFits,
    PlotSettings,
    QQPlot,
)

from scripts.plots import BarPlot, BarPlotWithError
from roos_pipeline.plots import EventMarker
from scripts.roos_plots import (
    BeforeAfterDensityPlot,
    DistributionDiffPlot,
    DotStripPlot,
    RidgelinePlot,
    RiseDeclinePlot,
    ViolinPlot,
)
from wa_analyzer.data import load_own_chat

## 4.6.1 Loading your data

Same rule as [02.5](../lesson2/roos_02.5-your_turn.ipynb): `load_own_chat()` raises when
there is no export loaded, on purpose — everything below is about *your* messages, not a
showcase dataset with a shape already built in.

In [ ]:
def canonical_author(name: str) -> str:
    """Louis appears under three names in the export (a contact rename mid-chat);
    group them into one author before any per-person distribution."""
    if "louis" in str(name).lower():
        return "Louis"
    return name


own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])
own["author"] = own["author"].apply(canonical_author)
own["n_chars"] = own["message"].str.len()
own.head()

## 4.6.2 What's already loaded

From `scripts/plots.py`: `BarPlot`, `BarPlotWithError` (02.2/02.4), `EventMarker` and
`RiseDeclinePlot` (03.3, the latter colouring a line by whether it's rising or falling).
None of them fit a distribution — that's `goad_toolkit`'s job, imported above:
`DistributionRegistry` + `DistributionFitter` do the fitting, `fit_table` ranks the
results, and `HistogramPlot` / `DistPlot` / `QQPlot` / `ECDFPlot` / `PlotFits` draw it.
All of them are `BasePlot`s too, so they compose with `plot_on` / `plot_on_axes` /
`create_figure(n_plots=...)` exactly like the classes in `scripts/plots.py` do — if a
chart this notebook needs doesn't exist in either place yet, a new `BasePlot` subclass in
`scripts/plots.py` is where it belongs, the same way `RiseDeclinePlot` got added there for
03.3.

## 4.6.3 Brainstorming with `goad`

Same tools as [02.5](../lesson2/roos_02.5-your_turn.ipynb#2.5.3), plus one built for
exactly this notebook:

- **`goad_distribution_method`** — the order of operations for fitting anything: count
  first, look before summarising, decide discrete-or-continuous, say what you expect
  *before* fitting, fit the whole registry rather than one family, and only then decide
  what counts as an outlier. Worth running before the first `DistributionFitter.fit()`
  call below, not after.
- **`goad_analysis_checklist`** — the six-stage interview (question, data, shape,
  encoding, critique, verification), one stage at a time, in your own words.
- **`goad_search`** / **`goad_list_concepts`** / **`goad_get_concept`** — look up a
  concept directly, e.g. `goad_get_concept('05')` for the full distributions chapter.
- **`goad_critique_visual`** — once there's a chart here to defend.

> **Your turn.**
>
> Before any `DistributionFitter.fit()` call — same order `goad_distribution_method`
> gives, steps 1-4 — answer these in your own words:
>
> 1. **Which variable, and why does it matter to you?** Not "what looks curious" but
>    what question about your own days or your own relationship it would actually answer.
> 2. **How many independent observations does it really rest on?** Messages are not
>    people — say both.
> 3. **Discrete or continuous?**
> 4. **What do you expect, before looking** — which family, and what mechanism would
>    produce it (a sum, a product, a steady rate, a rank)?
>
> **My answers:**
>
> 1. Response time — mine and Louis's, split before/after his move to NL (Nov 2024).
>    Before the move I might have had an overall lower response time and Louis a faster
>    one; after the move there might be a switch. Not sure if that's right — I'm used to
>    thinking about this in a time series, not a distribution.
> 2. Only the *close* response times should count — at night we sleep, and a reply after
>    hours of silence isn't really "a response time", it's "woke up". (Handled below by
>    excluding responses over a duration cutoff before doing anything else — see 4.6.4.)
> 3. Continuous (seconds).
> 4. Predicted the *outcome* (who's faster, before vs. after) above; haven't committed to
>    a specific family yet — that's still open.

## 4.6.4 Response time, before vs. after the move — first look, no fitting yet

Per §4.3 of the "five families" chapter: "comparing two groups by their histograms alone"
is one of distributions' classic ways of lying, when the groups differ in size — normalise,
or use ECDFs. This is that: four ECDFs (Roos/Louis × before/after) on one bin-free axis, no
family fitted yet — just looking at the shape, per `goad_distribution_method` step 2.

Two pieces reused rather than rebuilt:

- **`response_seconds`** — the same `ResponseTimeFeature` transform from `roos_01.4`:
  seconds to a message from the *other* author, `NaN` for same-author bursts (so the
  correlated-burst problem your own analysis-log already flagged for this variable stays
  visible, not fitted around).
- **`MOVE_DATE = "2024-11-01"`** — the same constant `roos_02.5` uses for Louis's move.

**One new choice, mine, not yours**: excluding responses over `MAX_RESPONSE_MINUTES`
(currently 4 hours) as "asleep, not a slow reply" — 04.4's own gap-fitting cell uses a
1-hour cutoff for *any* message gap ("a burst, not overnight"), but a response can take
longer than that and still be a real response, so 4 hours is more generous. Change the
constant and rerun if that doesn't match what you'd call "still a response".

In [ ]:
class ResponseTimeFeature(TransformBase):
    """Seconds to this message from the previous one, only when the author changed --
    i.e. how long the responder took to reply. Same-author bursts get NaN.
    (Unchanged from roos_01.4.)"""

    def transform(
        self,
        data: pd.DataFrame,
        timestamp_col: str = "timestamp",
        author_col: str = "author",
        feature: str = "response_seconds",
    ) -> pd.DataFrame:
        data = data.copy()
        ts = pd.to_datetime(data[timestamp_col], utc=True).dt.tz_convert(None)
        gap = ts.diff().dt.total_seconds()
        same_author = data[author_col] == data[author_col].shift(1)
        data[feature] = gap.where(~same_author)
        return data


MOVE_DATE = pd.Timestamp("2024-11-01")  # same constant as roos_02.5
MAX_RESPONSE_MINUTES = 240  # see the note above -- adjust and rerun to taste

own_response = Pipeline().add(ResponseTimeFeature).apply(own)
own_response["timestamp"] = own_response["timestamp"].dt.tz_convert(None)
own_response["period"] = np.where(
    own_response["timestamp"] < MOVE_DATE, "before move", "after move"
)

overnight_excluded = own_response["response_seconds"].gt(MAX_RESPONSE_MINUTES * 60).sum()
responses = own_response[
    own_response["response_seconds"].gt(0)
    & own_response["response_seconds"].le(MAX_RESPONSE_MINUTES * 60)
].copy()
responses["response_minutes"] = responses["response_seconds"] / 60

print(
    f"{overnight_excluded:,} responses over {MAX_RESPONSE_MINUTES / 60:.0f}h excluded "
    "(treated as asleep, not a slow reply)"
)
responses.groupby(["author", "period"])["response_minutes"].count()

In [ ]:
# Colour = person (grey-first isn't the point here -- no group is "the finding" yet,
# just a first look), linestyle = period, so before/after reads the same way for both
# people without needing four colours.
GROUP_STYLE = {
    ("Roos", "before move"): {"color": "#0072B2", "linestyle": "--"},
    ("Roos", "after move"): {"color": "#0072B2", "linestyle": "-"},
    ("Louis", "before move"): {"color": "#D55E00", "linestyle": "--"},
    ("Louis", "after move"): {"color": "#D55E00", "linestyle": "-"},
}

ecdf_settings = PlotSettings(
    figsize=(10, 5),
    title="Response time: Roos vs. Louis, before vs. after the move to NL",
    xlabel="response time (minutes)",
    ylabel="share of responses at or below",
)
host = ECDFPlot(ecdf_settings)
fig = ax = None
for i, ((author, period), style) in enumerate(GROUP_STYLE.items()):
    subset = responses.loc[
        (responses["author"] == author) & (responses["period"] == period), "response_minutes"
    ].to_numpy()
    label = f"{author}, {period}"
    if i == 0:
        fig, ax = host.plot(data=subset, label=label, **style)
    else:
        host.plot_on(ECDFPlot(ecdf_settings), data=subset, label=label, **style)
ax.legend(loc="lower right")

counts = responses.groupby(["author", "period"])["response_minutes"].count()
n_caption = "   ".join(f"{author}, {period}: n={counts[(author, period)]:,}" for author, period in GROUP_STYLE)
fig.text(0.5, -0.08, n_caption, ha="center", fontsize=8.5, color="dimgray")

## 4.6.5 What hour do we talk? Before vs. after the move

Not an event-discovery or pattern-discovery question — the event (`MOVE_DATE`) and the
prediction are both already named: a long-distance "good morning" / "sleep well" ritual
at wake-up and bedtime, before the move, that a shared apartment makes unnecessary after
it. That makes this a direct before/after comparison, same shape as 4.6.4, on a new
variable: which **hour** each message was sent in.

Two choices, both flagged rather than silent:

- **Night hours (00:00-04:59) excluded** — matches the "morning starts at 05:00" boundary
  already in `analysis-log.md`'s time-of-day buckets, so this reuses a definition you
  already settled on rather than inventing a new cutoff.
- **Share of each period's own messages, not raw counts** — the "before" and "after"
  periods have very different total volumes (§4.6.4 already showed that), and the five
  families chapter's own categories warning applies here too: proportions across unequal
  group sizes, or histograms with different sample sizes, mislead unless normalised.

**My prediction, before looking:** more messages in the evening before the move; more
messages during the day after the move.

In [ ]:
NIGHT_HOURS = range(0, 5)  # 00:00-04:59, matches analysis-log.md's morning-starts-at-05:00

with_hour = Pipeline().add(TimeFeatures, column="timestamp", features=["hour"]).apply(own_response)
daytime = with_hour[~with_hour["hour"].isin(NIGHT_HOURS)].copy()

hour_counts = daytime.groupby(["period", "hour"]).size().rename("n").reset_index()
hour_counts["share"] = hour_counts.groupby("period")["n"].transform(lambda s: s / s.sum())

hour_settings = PlotSettings(
    figsize=(13, 5),
    title='What hour do we talk? (night hours 00:00-04:59 excluded)',
    xlabel="hour of day",
    ylabel="share of that period's messages",
)
fig, ax = BarPlot(hour_settings).plot(
    data=hour_counts, x="hour", y="share", hue="period",
    hue_order=["before move", "after move"],
    palette={"before move": hour_settings.base_color, "after move": hour_settings.highlight_color},
)
ax.legend(title="")

n_caption = "   ".join(
    f"{period}: n={int(daytime[daytime.period == period].shape[0]):,}"
    for period in ["before move", "after move"]
)
fig.text(0.5, -0.1, n_caption, ha="center", fontsize=8.5, color="dimgray")

Read against the prediction above: evening vs. daytime hours, before vs. after — did the
weight shift the way you expected, and by how much?

**What I found:** (write it here)

## 4.6.6 Message length and burst length, before vs. after Benji

**Hypothesis, in your words**: since Benji arrived (11 May 2025) you send longer messages
and more bursts than Louis, and the gap between you two is bigger after the baby than
before.

This is the first place in the notebook that actually fits a family rather than just
looking at shape — `goad_distribution_method` steps 5-9, not just 1-4. That matters for
more than method: the graded rubric for this theme is explicit that "welke familie past,
en waarom hoort dat mechanisme bij jouw chat" is the actual ask — a histogram or ECDF
without a fitted family and a stated mechanism is "een plaatje, geen verhaal."

Two variables, one new date:

- **`BABY_DATE = "2025-05-11"`** — a different split than `MOVE_DATE` above; this section
  doesn't touch the move-based `period` column from 4.6.4/4.6.5.
- **Message length** — `n_chars`, already on `own`.
- **Burst length** — your choice: consecutive messages from the same author before the
  other one replies, i.e. a run-length on `author`. One number per *burst*, not per
  message — a 5-message burst is one observation, not five.

In [ ]:
BABY_DATE = pd.Timestamp("2025-05-11")

own["baby_period"] = np.where(
    own["timestamp"].dt.tz_convert(None) < BABY_DATE, "before baby", "after baby"
)

length_fitter = DistributionFitter(DistributionRegistry(), seed=42)
length_fits = {}
for author in ["Roos", "Louis"]:
    for period in ["before baby", "after baby"]:
        values = own.loc[
            (own["author"] == author) & (own["baby_period"] == period), "n_chars"
        ].to_numpy().astype(float)
        fits = length_fitter.fit(values, discrete=False)
        best = next(f for f in fits if isinstance(f, FitResult) and f.best_likelihood)
        length_fits[(author, period)] = (values, best)
        print(
            f"{author:6s} {period:11s}  {best.distribution:8s}  "
            f"mean {best.frozen_dist.mean():6.1f}  sd {best.frozen_dist.std():6.1f}  "
            f"(n={len(values):,})"
        )

In [ ]:
BABY_GROUP_STYLE = {
    ("Roos", "before baby"): {"color": "#0072B2", "linestyle": "--"},
    ("Roos", "after baby"): {"color": "#0072B2", "linestyle": "-"},
    ("Louis", "before baby"): {"color": "#D55E00", "linestyle": "--"},
    ("Louis", "after baby"): {"color": "#D55E00", "linestyle": "-"},
}

length_settings = PlotSettings(
    figsize=(10, 5),
    title="Message length: Roos vs. Louis, before vs. after Benji",
    xlabel="characters per message",
    ylabel="share of messages at or below",
)
host = ECDFPlot(length_settings)
fig = ax = None
for i, ((author, period), style) in enumerate(BABY_GROUP_STYLE.items()):
    values, best = length_fits[(author, period)]
    label = f"{author}, {period} ({best.distribution}, mean {best.frozen_dist.mean():.0f})"
    if i == 0:
        fig, ax = host.plot(data=values, label=label, **style)
    else:
        host.plot_on(ECDFPlot(length_settings), data=values, label=label, **style)
ax.legend(loc="lower right", fontsize=8.5)

n_caption = "   ".join(
    f"{author}, {period}: n={len(length_fits[(author, period)][0]):,}"
    for author, period in BABY_GROUP_STYLE
)
fig.text(0.5, -0.1, n_caption, ha="center", fontsize=8.5, color="dimgray")

**Why that family, if `lognorm` wins** (fill in against what the printout above actually
says): a message's length isn't a sum of independent contributions, it's closer to a
*product* — how much you have to say, multiplied by how rushed you are, multiplied by
whether you're mid-conversation or opening one, multiplied by topic complexity. 04.2 made
the identical argument for taxi fares (base fare × distance × traffic × surcharges) and it
predicted `lognorm` there too — the mechanism, not just the shape, is what makes this
prediction more than a coincidence. If a different family won instead, that mechanism
story needs to change to match it, not the other way around.

**Burst length next** — same treatment, but discrete: a *count* of consecutive messages,
not a continuous measurement.

In [ ]:
# One row per burst, not per message: consecutive same-author messages collapse into
# one run, counted once.
own_sorted = own.sort_values("timestamp").reset_index(drop=True)
run_id = (own_sorted["author"] != own_sorted["author"].shift()).cumsum()
bursts = (
    own_sorted.groupby(run_id)
    .agg(
        author=("author", "first"),
        timestamp=("timestamp", "first"),
        burst_length=("author", "size"),
    )
    .reset_index(drop=True)
)
bursts["baby_period"] = np.where(
    bursts["timestamp"].dt.tz_convert(None) < BABY_DATE, "before baby", "after baby"
)

burst_fitter = DistributionFitter(DistributionRegistry(), seed=42)
burst_fits = {}
for author in ["Roos", "Louis"]:
    for period in ["before baby", "after baby"]:
        values = bursts.loc[
            (bursts["author"] == author) & (bursts["baby_period"] == period), "burst_length"
        ].to_numpy().astype(float)
        fits = burst_fitter.fit(values, discrete=True)
        best = next(f for f in fits if isinstance(f, FitResult) and f.best_likelihood)
        burst_fits[(author, period)] = (values, best)
        print(
            f"{author:6s} {period:11s}  {best.distribution:8s}  "
            f"mean {best.frozen_dist.mean():5.2f}  sd {best.frozen_dist.std():5.2f}  "
            f"(n={len(values):,} bursts)"
        )

**How a burst is actually computed** — walking through the cell above:

1. **Sort by timestamp.** Order matters here: a burst is about who talked *in a row*.
2. **Mark every author-change.** `own_sorted["author"] != own_sorted["author"].shift()`
   is `True` exactly on the message where the author switches from the one before it.
3. **`.cumsum()` turns each change into the start of a new run.** A toy example makes
   this concrete — same mechanic, eight messages instead of 63,000:

   | # | author | changed from previous? | `run_id` (cumsum) |
   |---|---|---|---|
   | 1 | Roos | — (first row) | 1 |
   | 2 | Roos | False | 1 |
   | 3 | Louis | **True** | 2 |
   | 4 | Roos | **True** | 3 |
   | 5 | Roos | False | 3 |
   | 6 | Roos | False | 3 |
   | 7 | Louis | **True** | 4 |
   | 8 | Louis | False | 4 |

   Four runs: Roos×2, Louis×1, Roos×3, Louis×2. `run_id` only increases where the
   `True`/`False` column is `True` — everywhere else it repeats the same number, which
   is exactly what "still the same burst" means.
4. **`.groupby(run_id).agg(...)` collapses each run into one row**: which author, when
   it started (first timestamp — used for the before/after split), and `burst_length` =
   how many messages share that `run_id` (`.size()`).

One burst = one observation, whatever its length — a 7-message monologue counts once,
the same as a single lone message (`burst_length = 1`). That's why `bursts` has far
fewer rows (~12,500) than `own` (~63,000 messages): most bursts are more than one
message.

In [ ]:
burst_settings = PlotSettings(
    figsize=(10, 5),
    title="Burst length: Roos vs. Louis, before vs. after Benji",
    xlabel="messages per burst",
    ylabel="share of bursts at or below",
)
host = ECDFPlot(burst_settings)
fig = ax = None
for i, ((author, period), style) in enumerate(BABY_GROUP_STYLE.items()):
    values, best = burst_fits[(author, period)]
    label = f"{author}, {period} ({best.distribution}, mean {best.frozen_dist.mean():.2f})"
    if i == 0:
        fig, ax = host.plot(data=values, label=label, **style)
    else:
        host.plot_on(ECDFPlot(burst_settings), data=values, label=label, **style)
ax.legend(loc="lower right", fontsize=8.5)

n_caption = "   ".join(
    f"{author}, {period}: n={len(burst_fits[(author, period)][0]):,} bursts"
    for author, period in BABY_GROUP_STYLE
)
fig.text(0.5, -0.1, n_caption, ha="center", fontsize=8.5, color="dimgray")

**Why that family, if `nbinom` wins** (again: fill this in against the actual printout,
not this guess): a burst is "how many messages before the other person gets a word in" —
each message in the run either continues (you keep typing) or stops (you wait for a
reply), which is the same steady-continuation-probability mechanism 04.4 §4.4.1 used for
`nbinom` over `poisson`: not one fixed rate of continuing, but a rate that itself varies
by conversation, mood, or topic — a "sometimes I fire off ten in a row, sometimes just
one" pattern rather than a metronomic one. If `poisson` or something else wins instead,
that says the continuation rate is steadier than this story assumes, and the story should
say so.

**What I found:** (read both charts against your hypothesis — longer messages *and* more
bursts for you than Louis, with a bigger gap after Benji — write it here, including
anywhere the fitted family surprised you)

**Still open, if you want this to hold up under `spurious-correlations`**: everything
above is a fit and a picture, not yet a "how sure" check — `NullDistribution` (04.4
§4.4.3) would say whether the before/after gap in either variable is bigger than shuffling
the labels would produce on its own. Worth doing before this goes in the report.

## 4.6.7 How sure? A `NullDistribution` check on all four charts

Same idea for all four, so it's written once here rather than per chart: every comparison
above is "before vs. after, Roos vs. Louis" — a gap between two people that might itself
change between two periods. The statistic each check below measures is that **gap of
gaps** (also called an interaction, or difference-in-differences):

```
(Roos, after) − (Louis, after)   −   (Roos, before) − (Louis, before)
```

`NullDistribution` then shuffles the *before/after* label 2,000 times — keeping who-said-
what and each period's proportions exactly as they were, only breaking the link between
"when" and the value — and recomputes that same gap-of-gaps each time. If the real
gap-of-gaps sits far outside that shuffled cloud, "before/after" is doing real work; if it
sits comfortably inside, a random split would have produced a gap this big just as often,
and "there might be a switch" is not yet a finding.

`hour-of-day` has no author split (4.6.5 never had one), so its check is simpler: evening
share (hours 18:00-23:59, approximating the 17:30 boundary to whole hours) after the move
minus before it — no gap-of-gaps, just whether that shift is more than shuffle-noise.

In [ ]:
def response_gap_change(frame: pd.DataFrame) -> float:
    means = frame.groupby(["author", "period"])["response_minutes"].mean()
    return (means[("Roos", "after move")] - means[("Louis", "after move")]) - (
        means[("Roos", "before move")] - means[("Louis", "before move")]
    )


def evening_share_change(frame: pd.DataFrame) -> float:
    is_evening = frame["hour"].isin(EVENING_HOURS)
    shares = is_evening.groupby(frame["period"]).mean()
    return shares["after move"] - shares["before move"]


def length_gap_change(frame: pd.DataFrame) -> float:
    means = frame.groupby(["author", "baby_period"])["n_chars"].mean()
    return (means[("Roos", "after baby")] - means[("Louis", "after baby")]) - (
        means[("Roos", "before baby")] - means[("Louis", "before baby")]
    )


def burst_gap_change(frame: pd.DataFrame) -> float:
    means = frame.groupby(["author", "baby_period"])["burst_length"].mean()
    return (means[("Roos", "after baby")] - means[("Louis", "after baby")]) - (
        means[("Roos", "before baby")] - means[("Louis", "before baby")]
    )


EVENING_HOURS = range(18, 24)

response_null = NullDistribution(response_gap_change, n_iter=2000, seed=4).run(responses, label="period")
hour_null = NullDistribution(evening_share_change, n_iter=2000, seed=4).run(daytime, label="period")
length_null = NullDistribution(length_gap_change, n_iter=2000, seed=4).run(own, label="baby_period")
burst_null = NullDistribution(burst_gap_change, n_iter=2000, seed=4).run(bursts, label="baby_period")

for name, result in [
    ("response time gap-of-gaps (minutes)", response_null),
    ("evening-share shift", hour_null),
    ("message-length gap-of-gaps (chars)", length_null),
    ("burst-length gap-of-gaps (messages)", burst_null),
]:
    print(f"{name:38s} observed {result.observed:+7.3f}   two-sided p = {result.p_value():.4f}")

In [ ]:
null_settings = PlotSettings(
    figsize=(14, 9),
    title="Is the before/after gap bigger than shuffled labels would produce?",
    subplot_titles=[
        "Response time: (Roos−Louis) gap, after − before move",
        "Evening share (18:00-23:59), after − before move",
        "Message length: (Roos−Louis) gap, after − before baby",
        "Burst length: (Roos−Louis) gap, after − before baby",
    ],
    # Each panel's x-axis is the STATISTIC that got shuffled 2,000 times, not the raw
    # variable (not "minutes" or "characters" per message) -- naming that explicitly
    # here is the fix for "what is this axis even showing".
    subplot_xlabels=[
        "gap-of-gaps (minutes), one value per shuffle",
        "evening-share shift, one value per shuffle",
        "gap-of-gaps (characters), one value per shuffle",
        "gap-of-gaps (messages/burst), one value per shuffle",
    ],
    ylabel="number of shuffles landing here",
    max_cols=2,
)
host = NullPlot(null_settings)
fig, axes = host.create_figure(n_plots=4)
for ax, result in zip(axes, [response_null, hour_null, length_null, burst_null]):
    host.plot_on_axes(NullPlot(null_settings), ax, result=result)
    ax.legend(fontsize=8)

Read each panel the same way: grey cloud = what shuffling before/after produces on its
own; crimson line = what actually happened. Far outside the cloud = solid; inside it =
could be chance. The printed p-values above give the same read as a number — under 0.05
is a common (not sacred) line, and with 2,000 shuffles anything reported as "p < 0.0005"
is really "rarer than this test can resolve," not exactly zero.

**What I found:** (write it here, one line per variable — which of the four survived this
check, and which turned out to be a gap you can't distinguish from noise)

## 4.6.8 Burst length, Roos vs. Louis, side by side

Two panels instead of one ECDF: one per person, each with its own before/after fitted
curves overlaid. "Louis has fewer bursts than Roos" now reads as two panels sitting at
different heights; "Roos's bursts got longer after Benji" reads within her own panel —
both without a four-line legend to trace on one shared axis. `sharex`/`sharey` keep both
panels on the same scale, so comparing their heights directly is honest.

Curves, not histogram bars — the fit is already verified (4.6.6), and two overlapping
discrete histograms are exactly the clutter §7.3 of the visual-critique chapter warns
about; the smooth `pmf` curves are the same information with the mess removed.

In [ ]:
burst_compare_settings = PlotSettings(
    figsize=(12, 5),
    title="Mom takes the lead",
    subplot_titles=["Roos", "Louis"],
    xlabel="messages per burst",
    ylabel="probability",
    max_cols=2,
    sharex=True,
    sharey=True,
)
host = DistPlot(burst_compare_settings)
fig, axes = host.create_figure(n_plots=2)
for ax, author in zip(axes, ["Roos", "Louis"]):
    _, before_fit = burst_fits[(author, "before baby")]
    _, after_fit = burst_fits[(author, "after baby")]
    host.plot_on_axes(
        DistPlot(burst_compare_settings), ax, distribution=before_fit.frozen_dist,
        color=burst_compare_settings.base_color, marker="o",
        label=f"before baby ({before_fit.distribution}, mean {before_fit.frozen_dist.mean():.1f})",
    )
    host.plot_on_axes(
        DistPlot(burst_compare_settings), ax, distribution=after_fit.frozen_dist,
        color=burst_compare_settings.highlight_color, marker="o",
        label=f"after baby ({after_fit.distribution}, mean {after_fit.frozen_dist.mean():.1f})",
    )
    ax.legend(fontsize=8)

## 4.6.9 The real finding: not *more*, but *opposite directions*

Read against 4.6.8: the overall shift for either person is small, and that's the point
you're actually making — not "bigger bursts," but "yours grew and his shrank." Neither
the ECDF (4.6.6) nor the side-by-side panels (4.6.8) put that on the page directly; both
still ask the reader to compare two shapes and notice the direction themselves.

This chart does the subtraction for them — the same move 03.1 used on a time series
(`SubtractBaseline`: after minus before, so the residual is the plot), applied here to
two *fitted distributions* instead of two time windows. For each possible burst length,
`after.pmf(x) − before.pmf(x)`: positive means that length became *more* likely after
Benji, negative means *less* likely. One line per person, one shared zero line — Roos's
line should sit above zero further right (bigger bursts, more likely), Louis's below.

New class in `scripts/plots.py`: `DistributionDiffPlot` — draws a precomputed
`(x, diff)` pair as a line with the area shaded to the zero line, so the sign reads as
much from the fill as from the line itself.

In [ ]:
DIFF_COLORS = {"Roos": "#0072B2", "Louis": "#D55E00"}

# Shared x-range: covers the 99th percentile of every fitted distribution, so neither
# person's curve gets cut off, and both are drawn on the same integers for a fair
# subtraction.
max_x = max(
    int(np.ceil(fit.frozen_dist.ppf(0.99)))
    for _, fit in burst_fits.values()
)
x = np.arange(1, max_x + 1)

diff_settings = PlotSettings(
    figsize=(10, 5),
    title="Mom takes the lead",
    xlabel="messages per burst",
    ylabel="P(after Benji) − P(before Benji)",
)
host = DistributionDiffPlot(diff_settings)
fig = ax = None
for i, author in enumerate(["Roos", "Louis"]):
    _, before_fit = burst_fits[(author, "before baby")]
    _, after_fit = burst_fits[(author, "after baby")]
    diff = after_fit.frozen_dist.pmf(x) - before_fit.frozen_dist.pmf(x)
    if i == 0:
        fig, ax = host.plot(x=x, diff=diff, color=DIFF_COLORS[author], label=author)
    else:
        host.plot_on(DistributionDiffPlot(diff_settings), x=x, diff=diff,
                     color=DIFF_COLORS[author], label=author)
host.plot_on(HorizontalLine(diff_settings), y=0)
ax.legend()

fig.text(
    0.5, -0.08,
    "Positive = that burst length became more likely after Benji; negative = less "
    "likely. Fitted pmf difference, not raw counts.",
    ha="center", fontsize=8.5, color="dimgray",
)

## 4.6.10 Does the pregnancy window stand out? Nausea, fatigue, and both combined

Three questions, one recipe, reused three times:

1. **Nausea** — a comparison chart and a distribution chart, since that's what was
   asked for both ways.
2. **Fatigue** — the distribution chart alone.
3. **Nausea + fatigue combined** — same distribution chart, on "mentions either
   symptom," to answer directly: is the pregnancy window recognisable against the rest
   of the chat?

**Same categories as `roos_03.3`**, unchanged (`NAUSEA_PATTERN`, `FATIGUE_PATTERN`,
English/Dutch/French each). **Same pregnancy window**, unchanged (`PREGNANCY_START` =
`BABY_DATE` − 41 weeks, the last-period due-date convention from `roos_03.3`).

New here: instead of a weekly *line* (03.3's approach), each category becomes a
**weekly count**, fitted with `DistributionFitter` and split **during pregnancy** vs.
**the rest of the chat** — the same "messages per day" recipe goal 4.17 asks for, at
weekly resolution because nausea alone is rare enough (n=38 all-time) that daily counts
would be almost entirely zero either way.

Both chart types are `scripts/plots.py` classes: `BarPlot` for the comparison,
`DistributionDiffPlot` (4.6.9) for the distribution — reused as-is, not rebuilt.

In [ ]:
# Nausea/fatigue patterns and PREGNANCY_START, unchanged from roos_03.3.
NAUSEA_PATTERN = (
    r"(?i)("
    r"\bnausea(ted|ous)?\b|\bqueasy\b|"
    r"misselijk(heid)?|"
    r"naus[ée]e?(ux|se)?|mal au c[oœ]eur"
    r")"
)
FATIGUE_PATTERN = (
    r"(?i)("
    r"\bfatigue(d)?\b|\btired\b|\bexhaust(ed|ion)\b|\bworn out\b|"
    r"\bmoe\b|\bmoeheid\b|\bvermoeidheid\b|\bvermoeid\b|\buitgeput\b|"
    r"fatigu[ée]e?|[ée]puis[ée]e?|crev[ée]e?"
    r")"
)
PREGNANCY_START = BABY_DATE - pd.Timedelta(weeks=41)

symptoms = (
    Pipeline()
    .add(RegexFeature, name="nausea_words", column="message", pattern=NAUSEA_PATTERN,
         feature="n_nausea_words", mode="count")
    .add(RegexFeature, name="fatigue_words", column="message", pattern=FATIGUE_PATTERN,
         feature="n_fatigue_words", mode="count")
    .apply(own)
)
symptoms["nausea"] = symptoms["n_nausea_words"].gt(0)
symptoms["fatigue"] = symptoms["n_fatigue_words"].gt(0)
symptoms["either"] = symptoms["nausea"] | symptoms["fatigue"]
symptoms["timestamp_naive"] = symptoms["timestamp"].dt.tz_convert(None)

print(symptoms[["nausea", "fatigue", "either"]].sum().rename("messages mentioning it"))


def weekly_in_out(frame: pd.DataFrame, flag_col: str) -> pd.DataFrame:
    """Weekly count of messages where flag_col is True, reindexed to the chat's full
    weekly range (quiet weeks are zero, not missing), labelled by whether that week
    falls inside the pregnancy window.

    The full range comes from resampling the UNFILTERED frame with the same "W" rule
    (not a hand-built `pd.date_range(min, max, freq="W")` -- that keeps whichever
    time-of-day the first raw message happened to carry, which never lines up with
    `resample`'s own midnight-Sunday bins, and silently reindexes every real count to
    zero). `rename_axis("week")` before `reset_index()` names the resulting column
    directly, rather than relying on `reset_index()`'s own naming -- which follows the
    Series' index name (inherited from `set_index("timestamp_naive")` further up, not
    the generic "index"), and renaming the wrong column name is a silent no-op, not an
    error, which is exactly what let this bug through once already.
    """
    weekly = frame[frame[flag_col]].set_index("timestamp_naive").resample("W").size()
    full_range = frame.set_index("timestamp_naive").resample("W").size().index
    weekly = (
        weekly.reindex(full_range, fill_value=0)
        .rename("n")
        .rename_axis("week")
        .reset_index()
    )
    weekly["in_pregnancy"] = weekly["week"].between(PREGNANCY_START, BABY_DATE)
    return weekly


def fit_in_out(weekly: pd.DataFrame) -> dict:
    """Fit the discrete registry separately inside vs. outside the pregnancy window.
    Falls back to a plain poisson (always fittable, even on near-all-zero data) when
    the registry search finds nothing usable -- reported explicitly, not hidden."""
    fitter = DistributionFitter(DistributionRegistry(), seed=42)
    fits = {}
    for label, is_in in [("outside pregnancy", False), ("during pregnancy", True)]:
        values = weekly.loc[weekly["in_pregnancy"] == is_in, "n"].to_numpy().astype(float)
        candidates = fitter.fit(values, discrete=True)
        best = next(
            (f for f in candidates if isinstance(f, FitResult) and f.best_likelihood), None
        )
        if best is None:
            best = fitter.fit_distribution("poisson", values)
            print(f"{label:18s} registry fit found nothing usable -- fell back to poisson")
        fits[label] = (values, best)
        print(
            f"{label:18s} {best.distribution:8s} mean {best.frozen_dist.mean():5.2f}  "
            f"sd {best.frozen_dist.std():5.2f}  (n={len(values)} weeks)"
        )
    return fits


def plot_pregnancy_diff(fits: dict, title: str) -> tuple:
    """DistributionDiffPlot of P(during pregnancy) - P(outside), same recipe as 4.6.9."""
    out_values, out_fit = fits["outside pregnancy"]
    in_values, in_fit = fits["during pregnancy"]
    max_x = max(
        int(np.ceil(out_fit.frozen_dist.ppf(0.99))),
        int(np.ceil(in_fit.frozen_dist.ppf(0.99))),
        1,
    )
    x = np.arange(0, max_x + 1)
    diff = in_fit.frozen_dist.pmf(x) - out_fit.frozen_dist.pmf(x)

    settings = PlotSettings(
        figsize=(10, 5),
        title=title,
        xlabel="mentions per week",
        ylabel="P(during pregnancy) − P(outside pregnancy)",
    )
    host = DistributionDiffPlot(settings)
    fig, ax = host.plot(x=x, diff=diff, label="pregnancy − rest of chat",
                        color=settings.highlight_color)
    host.plot_on(HorizontalLine(settings), y=0)
    ax.legend()
    fig.text(
        0.5, -0.08,
        f"During pregnancy: {in_fit.distribution}, mean {in_fit.frozen_dist.mean():.2f}/week "
        f"(n={len(in_values)} weeks). Outside: {out_fit.distribution}, mean "
        f"{out_fit.frozen_dist.mean():.2f}/week (n={len(out_values)} weeks).",
        ha="center", fontsize=8.5, color="dimgray",
    )
    return fig, ax

### Nausea — comparison, then distribution

In [ ]:
nausea_weekly = weekly_in_out(symptoms, "nausea")
nausea_fits = fit_in_out(nausea_weekly)

nausea_summary = (
    nausea_weekly.groupby("in_pregnancy")["n"].mean().rename("mean_per_week").reset_index()
)
nausea_summary["period"] = nausea_summary["in_pregnancy"].map(
    {True: "during pregnancy", False: "outside pregnancy"}
)

nausea_bar_settings = PlotSettings(
    figsize=(7, 4.5),
    title="Nausea mentions: weekly average, in vs. outside the pregnancy window",
    xlabel="",
    ylabel="mean mentions per week",
)
fig, ax = BarPlot(nausea_bar_settings).plot(
    data=nausea_summary, x="period", y="mean_per_week",
    order=["outside pregnancy", "during pregnancy"],
    palette={
        "outside pregnancy": nausea_bar_settings.base_color,
        "during pregnancy": nausea_bar_settings.highlight_color,
    },
)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

In [ ]:
plot_pregnancy_diff(nausea_fits, "Nausea: does the pregnancy window stand out?")

### Fatigue — distribution only

In [ ]:
fatigue_weekly = weekly_in_out(symptoms, "fatigue")
fatigue_fits = fit_in_out(fatigue_weekly)
plot_pregnancy_diff(fatigue_fits, "Fatigue: does the pregnancy window stand out?")

### Nausea + fatigue combined — is pregnancy recognisable?

In [ ]:
either_weekly = weekly_in_out(symptoms, "either")
either_fits = fit_in_out(either_weekly)
plot_pregnancy_diff(either_fits, "Nausea + fatigue combined: is pregnancy recognisable?")

## 4.6.11 Message length per topic: ridgeline and violin

**Hypothesis, in your words:** messages about `planning_logistics` are longer than
messages from the other categories.

**Same categories as `roos_01.4`**, patterns copied over unchanged: love, financial,
child, climbing, running, food, planning_logistics. Two things to know before reading
the charts:

- **`food` is commented out in `roos_01.4`**, with no reason logged in
  `analysis-log.md`. It's back in here because you asked for it. If you dropped it on
  purpose (e.g. too many false positives from `courses`/`pasta`/`eggs`), drop it here too.
- **`be_careful` is left out**, same as you already did in 01.4's small multiples, and
  it wasn't on this list anyway.

Three choices, mine, flagged so you can overrule them:

- **One message can count in several categories.** "Can you pay the daycare when you're
  home?" counts under `financial`, `child` *and* `planning_logistics`. About 13% of
  categorised messages hit more than one. The alternative (one label per message) needs
  a tie-break rule you'd have to invent and defend.
- **An `all messages` reference row, in grey at the bottom.** Without it you can't tell
  "planning messages are long" from "*any* message that happens to contain a keyword is
  long". Delete it if you disagree, but read the tip at the end first.
- **Log x-axis.** 4.6.6 already showed message length has a long right tail. On a
  linear axis every ridge piles up against zero and the differences disappear. The
  black tick in each ridge is that group's median.

`planning_logistics` is the one in colour, because it's your prediction (grey-first,
same rule as 02.2). The rows are sorted by median, so the order itself answers
"which topic is longest".

In [ ]:
# Patterns copied unchanged from roos_01.4 (food un-commented, be_careful left out).
TOPIC_PATTERNS: dict[str, str] = {
    "love": (
        r"(?i)("
        r"je\s*t'?aime|tu me manques|mon amour|mon c[oœ]ur|ma belle|"
        r"mon ch[ée]ri|ma ch[ée]rie|je pense [aà] toi|bisous?|t'es (belle|beau)|"
        r"ik houd? van (je|jou)|ik mis (je|jou)|schatje|liefje|"
        r"i love you|i miss you|my love|sweetheart|darling|"
        r"❤️|😘|🥰|😍|💕|💖|💗|💓|💞|😻|💋"
        r"|je pense a toi|ik denk aan (je|jou)"
        r")"
    ),
    "financial": (
        r"(?i)("
        r"hypotheek(adviseur)?|hypoth[eè]que|mortgage( advisor)?|"
        r"rekening|compte( ?[eé]pargne)?|bank account|"
        r"belasting|imp[oô]ts?|\btaxe?s?\b|salaris|salaire|\bsalary\b|\bwage\b|"
        r"loon|huur|loyer|\brent\b|borg|caution|deposit|"
        r"spaarrekening|savings account|\bgeld\b|argent|\bmoney\b|"
        r"betalen|payer|\bpay\b|overmaken|virer|virement|transfer|"
        r"bunq|asn|\bing\b|rabobank|funda|velduil|"
        r"notaris|notaire|\bnotary\b|makelaar|agent immobilier|real estate agent|"
        r"iban|factuur|facture|invoice|budget"
        r")"
    ),
    "child": (
        r"(?i)("
        r"benji|b[ée]b[ée]|\bbaby\b|cr[eè]che|kinderdagverblijf|opvang|daycare|partou|basker|kinderdagopvang|"
        r"r[eé]cuperer|r[eé]cupes|prendre soin de|"
        r"\benfant\b|biberon|bottle[- ]?feeding|sucette|fopspeen|speentje|"
        r"pacifier|dummy|allaiter|donner les? seins?|borstvoeding|breastfeeding|t[ée]t[ée]e|"
        r"luiers?|diapers?|couches?|doekje|hydrophile|vitamines|hydrofiele doek|\bwipe\b|"
        r"kinderwagen|poussette|stroller|pram|la remorque|bike trailer|"
        r"consultatiebureau|kraamzorg|zwangerschap|grossesse|pregnan(cy|t)|"
        r"verloskundige|sage-femme|midwife|kolven|[ée]chographie"
        r")"
    ),
    "climbing": (
        r"(?i)("
        r"escalad(er|e)|salle de grimpe|grimpe|grimper|\bclimb(ing)?\b|climbing gym|"
        r"klimmen|klimmuur( centraal)?|\bboulder\b|\bbloc\b|\bvoie\b|"
        r"arkose|mountain network|\bncc\b|neoliet|\b[4-8][abc]\+?\b"
        r")"
    ),
    "running": (
        r"(?i)("
        r"\brun(ning)?\b|courir|\btrail\b|\bbornes?\b|kilom[eè]tre|\bkm\b|marathon|"
        r"hardlopen|hardloopschoenen|joggen|strava|parkrun|\b10k\b|\btempo\b|"
        r"chaussures|\bshoes\b"
        r")"
    ),
    "food": (
        r"(?i)("
        r"\beten\b|manger|\bd[iî]ner\b|avondeten|d[eé]jeuner|petit-d[eé]jeuner|\blunch\b|"
        r"ontbijt|breakfast|\bcook(ing)?\b|cuisiner|koken|recette|recept|recipe|"
        r"restaurant|\bhonger\b|\bfaim\b|hungry|dalle|"
        r"l[eé]gumes?|carottes?|courgettes?|poireaux?|oignons?|tomates?|pommes? de terre|"
        r"\bsalade\b|[ée]pinards?|brocolis?|poivrons?|champignons?|choux?|aubergines?|haricots?|"
        r"groenten|wortel(en)?|\bprei\b|uien?|tomaten?|aardappel(en)?|\bsla\b|spinazie|"
        r"paprika('s)?|paddenstoel(en)?|\bkool\b|bonen|"
        r"vegetables?|carrots?|zucchini|leeks?|onions?|tomatoe?s?|potatoe?s?|lettuce|spinach|"
        r"broccoli|peppers?|mushrooms?|cabbage|eggplant|beans?|"
        r"p[aâ]tes|pasta|œufs?|oeufs?|eggs?|\blait\b|\bmilk\b|melk|"
        r"fromage blanc|kwark|chocolat|chocolade|chocolate"
        r")"
    ),
    "planning_logistics": (
        r"(?i)("
        r"\bquand\b|je suis l[aà]|j'arrive|arriver|tu rentres quand|je rentre|en route|je pars|"
        r"[aà] quelle heure|hoe laat|\bwhen\b|i'll be there|\bthuis\b|\bhome\b|t'es partie?|"
        r"onderweg|on the way|\btrein\b|\btrain\b|vlucht|\bflight\b|\bagenda\b|"
        r"rendez-vous|afspraak|appointment|"
        r"boodschappen|courses?|lidl"
        r")"
    ),
}
ALL_LABEL = "all messages"

topic_pipeline = Pipeline()
for topic, pattern in TOPIC_PATTERNS.items():
    topic_pipeline.add(
        RegexFeature, name=f"{topic}_topic", column="message", pattern=pattern,
        feature=f"n_{topic}_topic", mode="count",
    )
with_topics = topic_pipeline.apply(own)

# Long format: one row per (message, topic it matches), so a message that hits two
# topics appears twice -- plus every message once more under ALL_LABEL.
topic_lengths = pd.concat(
    [
        with_topics.loc[with_topics[f"n_{topic}_topic"].gt(0), ["n_chars"]].assign(topic=topic)
        for topic in TOPIC_PATTERNS
    ]
    + [with_topics[["n_chars"]].assign(topic=ALL_LABEL)],
    ignore_index=True,
)

length_summary = (
    topic_lengths.groupby("topic")["n_chars"]
    .agg(n="size", median="median", mean="mean", p90=lambda s: s.quantile(0.9))
    .sort_values("median", ascending=False)
)
# Topics sorted by median, longest on top; the reference row always last.
topic_order = [t for t in length_summary.index if t != ALL_LABEL] + [ALL_LABEL]
topic_labels = {t: f"{t} (n={length_summary.loc[t, 'n']:,})" for t in topic_order}
length_summary.loc[topic_order].round(1)

In [ ]:
ridge_settings = PlotSettings(
    figsize=(10, 7),
    title="Message length per topic (black tick = median)",
    xlabel="characters per message (log scale)",
    ylabel="",
)
fig, ax = RidgelinePlot(ridge_settings).plot(
    data=topic_lengths, x="n_chars", group="topic",
    order=topic_order, labels=topic_labels, highlight_groups=["planning_logistics"],
)
fig.text(
    0.5, -0.04,
    "Each ridge scaled to the same height: compares shape, not how many messages a topic has. "
    "A message matching two topics is counted in both.",
    ha="center", fontsize=8.5, color="dimgray",
)

In [ ]:
violin_settings = PlotSettings(
    figsize=(10, 7),
    title="Message length per topic (dashed lines = quartiles, middle one = median)",
    xlabel="characters per message (log scale)",
    ylabel="",
)
fig, ax = ViolinPlot(violin_settings).plot(
    data=topic_lengths, x="n_chars", group="topic",
    order=topic_order, labels=topic_labels, highlight_groups=["planning_logistics"],
)
fig.text(
    0.5, -0.04,
    "Same data as the ridgeline. Each violin has the same maximum width, so width "
    "compares shape, not topic size.",
    ha="center", fontsize=8.5, color="dimgray",
)

Two charts, same data, two different questions:

- **Ridgeline**: where does each topic's *peak* sit, and does any topic have a
  second hump (e.g. a pile of one-emoji replies under `love`)?
- **Violin**: how do the medians and middle 50% line up? The quartile lines make
  "longer" a comparison you can read off the chart, not just an impression.

**My prediction, before looking:** `planning_logistics` messages are longer than the
other categories.

**What I found:** (write it here: where does `planning_logistics` actually rank, and
how far is *every* topic from the grey `all messages` row?)

## 4.6.12 Graded visualisation 3: a distribution before and after the positive test

**The assignment (goal 4.17):** *"de eigen chat splitsen op een bekend event (uit 03.3),
berichten per dag en gaps binnen bursts aan beide kanten fitten, en het resultaat opschrijven
als zin met parameters erin."* And the theme's bar: *"welke familie past, en waarom hoort dat
mechanisme bij jouw chat?"* So every fit below comes with a family **and** a mechanism.

**My hypothesis**, from earlier exploration: *the pregnancy changed what we talked about, not
how we talked.* Two layers test it:

1. **How we talked** (the rhythm): gaps between messages inside a burst, and messages per day.
   Expectation: barely any change.
2. **What we talked about**: messages that mention fatigue or nausea. Expectation: a clear rise.

**Focus.** Messages per day is required by the assignment and is fitted below, but it is a
side layer: how many messages we send is not what this story is about. The weight sits on the
gaps within bursts and on the content, and §4.6.13 zooms in on the part that matters most to
me: the hard weeks of the first trimester.

**Event and windows.** The positive pregnancy test, **29 August 2024**, with **77 days (11
weeks) on each side**: before = 13 June – 28 August, after = 29 August – 13 November 2024. Both
windows sit inside the long-distance period (Louis in Paris, me in the Netherlands; he moved to
Utrecht mid-November 2024), which keeps the move out of the comparison. Whole weeks on both
sides also means both windows hold the same number of Mondays, Saturdays, and so on.

**Choices taken from my own notebooks, not invented here:**

- **Event date `2024-08-29`**, verified in the chat itself, and the same as `my_event` in
  `roos_04.4`. `roos_03.3`'s marker for the test sits on `2024-09-01`, three days late: that
  date is actually the start of pregnancy week 5 (see §4.6.13), so the 03.3 marker is the one
  to correct.
- **A burst ends at a gap of more than 60 minutes**, any sender: the same 1-hour cut `roos_04.4`
  uses for its gap fit ("a burst, not overnight"). Note this is a *different* "burst" from
  §4.6.6, where a burst is a run of messages from one author. 15 and 30 minutes are tried in
  the robustness check.
- **Colours**: grey for *before*, crimson for *after*, in every chart below: the grey-first rule
  from 02.2, where the side that carries the change gets the colour.

In [ ]:
EVENT_DATE = pd.Timestamp("2024-08-29")  # positive pregnancy test -- same as roos_04.4's my_event
WINDOW_DAYS = 77  # 11 weeks each side: whole weeks, so both sides hold the same weekdays
BURST_GAP_MINUTES = 60  # same 1-hour cut roos_04.4 uses for "a burst, not overnight"
NIGHT_START_HOURS = range(0, 7)  # cleaning only: drop gaps that START between 00:00 and 06:59
SIDES = ["before", "after"]
SIDE_COLORS = {"before": "#7f7f7f", "after": PlotSettings().highlight_color}  # grey-first, 02.2

FIGURES = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()) / "figures"
FIGURES.mkdir(exist_ok=True)


def save(fig, name: str) -> None:
    """PNG for the report, SVG for anything that needs to scale."""
    for suffix in ("png", "svg"):
        fig.savefig(FIGURES / f"{name}.{suffix}", dpi=200, bbox_inches="tight")


# Timestamps arrive labelled UTC but are the phone's local clock (almost nothing between
# 02:00 and 05:00), so dropping the label gives local wall-clock time -- same as 4.6.4.
chat = own[["timestamp", "author", "message"]].copy()
chat["ts_local"] = chat["timestamp"].dt.tz_convert(None)
chat = chat.sort_values("ts_local").reset_index(drop=True)


def window(days: int = WINDOW_DAYS) -> tuple[pd.Timestamp, pd.Timestamp]:
    """[start, end): `days` days before the event and `days` days from it onwards."""
    return EVENT_DATE - pd.Timedelta(days=days), EVENT_DATE + pd.Timedelta(days=days)


start, end = window()
print(f"before: {start.date()} to {(EVENT_DATE - pd.Timedelta(days=1)).date()}")
print(f"after:  {EVENT_DATE.date()} to {(end - pd.Timedelta(days=1)).date()}")
print(f"{chat['ts_local'].between(start, end, inclusive='left').sum():,} messages in the two windows")

### Step 1 — messages per day (required; a side layer)

**What:** count the messages on every calendar day in each window, *including days with zero
messages* (a silent day is an observation, not a missing one). Then fit two families by maximum
likelihood and compare them with AIC (`2k − 2·log-likelihood`; lower is better, and a
difference of more than about 10 is decisive).

- **Poisson**: messages arrive at one steady rate μ. It makes a checkable promise: variance =
  mean, so variance/mean ≈ 1.
- **Negative binomial**: a Poisson whose rate itself changes from day to day. Two parameters:
  the mean μ and a shape r (or dispersion α = 1/r). Small r = the daily rate wanders a lot.

**Why not `DistributionFitter` here:** it passes `nbinom` to `scipy.stats.fit`, which only
tries whole-number values for r. A dispersion of 1.4 would come back as 1, so the fit is done
directly with `scipy.optimize` instead (same maximum-likelihood idea, r allowed to be any
positive number).

In [ ]:
def messages_per_day(days: int = WINDOW_DAYS) -> pd.DataFrame:
    """One row per calendar day in the window, zero-message days included."""
    lo, hi = window(days)
    in_window = chat[(chat["ts_local"] >= lo) & (chat["ts_local"] < hi)]
    all_days = pd.date_range(lo, hi - pd.Timedelta(days=1), freq="D")
    counts = (
        in_window.groupby(in_window["ts_local"].dt.normalize()).size()
        .reindex(all_days, fill_value=0)
        .rename("messages").rename_axis("day").reset_index()
    )
    counts["side"] = np.where(counts["day"] < EVENT_DATE, "before", "after")
    return counts


def aic(log_likelihood: float, k: int) -> float:
    return 2 * k - 2 * log_likelihood


def nbinom_from(mu: float, r: float):
    """scipy's nbinom takes (n, p); n is r, and p = r / (r + mu)."""
    return stats.nbinom(r, r / (r + mu))


def fit_daily(values: np.ndarray) -> dict:
    """Poisson and negative binomial by maximum likelihood, compared on AIC.

    Not via DistributionFitter: it hands nbinom to scipy.stats.fit, which only tries
    whole-number r -- a dispersion of 2.3 would come back as 2.
    """
    poisson_mu = values.mean()  # the Poisson MLE is the sample mean
    poisson_ll = stats.poisson.logpmf(values, poisson_mu).sum()

    def nll(theta):  # log-parameters keep mu and r positive
        return -nbinom_from(np.exp(theta[0]), np.exp(theta[1])).logpmf(values).sum()

    res = optimize.minimize(nll, [np.log(values.mean()), 0.0], method="Nelder-Mead",
                            options={"xatol": 1e-8, "fatol": 1e-8})
    mu, r = np.exp(res.x)
    return {
        "n_days": len(values),
        "mean": values.mean(),
        "vmr": values.var(ddof=1) / values.mean(),
        "poisson_mu": poisson_mu,
        "poisson_aic": aic(poisson_ll, 1),
        "nb_mu": mu,
        "nb_r": r,
        "nb_alpha": 1 / r,
        "nb_aic": aic(-res.fun, 2),
        "nb_dist": nbinom_from(mu, r),
    }


per_day = messages_per_day()
daily = {side: fit_daily(per_day.loc[per_day["side"] == side, "messages"].to_numpy().astype(float))
         for side in SIDES}

for side in SIDES:
    d = daily[side]
    print(f"{side:6s} {d['n_days']} days   observed mean {d['mean']:5.1f}   variance/mean {d['vmr']:5.1f}")
    print(f"       poisson  mu = {d['poisson_mu']:5.1f}                      AIC {d['poisson_aic']:8.1f}")
    print(f"       nbinom   mu = {d['nb_mu']:5.1f}   r = {d['nb_r']:.2f}   alpha = {d['nb_alpha']:.2f}   AIC {d['nb_aic']:8.1f}"
          f"   (dAIC poisson - nbinom = {d['poisson_aic'] - d['nb_aic']:,.0f})")

**What this means.**

- **Poisson is rejected on both sides, by a wide margin**: ΔAIC ≈ 2,400. Variance/mean is
  about 32 on both sides, where Poisson promises 1. That rejection *is* a finding: there is no
  steady daily rate in this chat.
- **The negative binomial is the family**, on both sides.
- **The mean barely moved:** μ ≈ 85 before, ≈ 96 after.
- **The dispersion moved more than the mean:** r = 1.38 before and 2.02 after, so the
  before-days were *more* erratic. The robustness check below shows where that comes from.

**Mechanism: why a negative binomial belongs to this chat.** A negative binomial is a
*gamma-Poisson mixture*. Each day has its own rate, drawn from a spread of possible rates
(the gamma part). Given that rate, messages arrive at random moments (the Poisson part).
For us, what sets a day's rate is easy to name:

- a workday or a weekend;
- whether there are plans to make;
- mood;
- above all, whether we are *apart* (long-distance, so the chat is the relationship) or
  *together* on a visit (and the chat goes nearly quiet).

A day is not "85 messages give or take a few". It is "a rate of about 85 that routinely
halves or doubles", which is exactly what r ≈ 1.4–2 says.

One more thing worth knowing: before the test, the variance/mean check and the fitted r don't
agree as well as after. That hints the before window holds two kinds of day (ordinary days and
near-empty ones). See the weekly table in the robustness section.

In [ ]:
daily_settings = PlotSettings(
    figsize=(10, 5),
    title="",
    xlabel="messages per day",
    ylabel="share of days, per message/day",
)
top = int(np.ceil(per_day["messages"].max() / 20) * 20)
fig, ax = BeforeAfterDensityPlot(daily_settings).plot(
    samples={side: per_day.loc[per_day["side"] == side, "messages"].to_numpy() for side in SIDES},
    bins=np.arange(0, top + 20, 20),
    colors=SIDE_COLORS,
    densities={side: daily[side]["nb_dist"].pmf for side in SIDES},
    discrete=True,
    labels={
        side: f"{side} the test: μ = {daily[side]['nb_mu']:.0f}, r = {daily[side]['nb_r']:.1f}\n"
              f"(negative binomial, {daily[side]['n_days']} days)"
        for side in SIDES
    },
    label_positions={"before": (0.55, 0.92), "after": (0.55, 0.76)},
)
ax.set_title(
    f"Messages per day: a small rise (μ {daily['before']['nb_mu']:.0f} → {daily['after']['nb_mu']:.0f}), "
    f"inside the day-to-day spread",
    loc="left", fontsize=12,
)
fig.text(0.01, -0.04, "Bars: observed days (bins of 20 messages). Curves: fitted negative binomial. "
         "Both windows fall in the long-distance period, before the move to Utrecht.",
         fontsize=8.5, color="dimgray")
save(fig, "vis3_messages_per_day")

Bars and curves share one density scale (share of days per message/day), so the eye can
check the fit. The grey curve peaks further left than the crimson one, pulled there by the
handful of near-empty days before the test.

**The hypothesis on this layer:** the rise from μ 85 to 96 is small next to the day-to-day
spread. The shuffle test in step 3 checks whether a random split of the same days would
produce a gap this size anyway.

### Step 2 — gaps within bursts

**What:** the seconds between two consecutive messages, whoever sent them, as long as the gap
is at most the burst threshold (60 minutes). A gap belongs to the side where its first message
was sent.

**Cleaning, not a finding:** gaps that *start* between 00:00 and 06:59 are left out, so a
"good night … good morning" pair doesn't enter as a 7-hour gap. This is only there to keep
sleep out of the measurement. It is not part of the story, and the story below never leans
on it.

**Families compared on AIC:** exponential, lognormal and Weibull, each with location fixed at 0.
Plus a **power-law tail**, p(x) ∝ x^−α above a cut-off x_min. x_min is the value that makes
the fitted tail closest to the data (the smallest Kolmogorov-Smirnov distance, the standard
Clauset-Shalizi-Newman recipe), and α is fitted by maximum likelihood.

**Truncation:** a gap inside a burst *can't* be longer than 60 minutes; the burst definition
cuts it off. A family fitted as if the tail continued beyond that would be judged on data it
was never allowed to see. So every fit here is **truncated at the threshold**: the density is
re-normalised to add up to 1 over the range the data can actually take. (Timestamps are whole
seconds, so the shortest possible gap is 1 s; the very first bins are coarse because of that.)

In [ ]:
def within_burst_gaps(lo: pd.Timestamp, hi: pd.Timestamp, burst_minutes: int = BURST_GAP_MINUTES) -> pd.DataFrame:
    """Seconds between consecutive messages (any sender) inside a burst, labelled by side.

    A gap belongs to the window its first message falls in. Kept: 0 < gap <= burst_minutes.
    Dropped (cleaning, not a finding): gaps starting between 00:00 and 06:59.
    """
    gap = chat["ts_local"].diff().dt.total_seconds()
    started = chat["ts_local"].shift()
    frame = pd.DataFrame({"gap": gap, "started": started}).dropna()
    keep = (
        frame["gap"].gt(0)
        & frame["gap"].le(burst_minutes * 60)
        & ~frame["started"].dt.hour.isin(NIGHT_START_HOURS)
        & frame["started"].ge(lo)
        & frame["started"].lt(hi)
    )
    out = frame[keep].copy()
    out["side"] = np.where(out["started"] < EVENT_DATE, "before", "after")
    return out


# Continuous families, all fitted TRUNCATED to (0, upper]: a burst can't contain a gap longer
# than the burst threshold, so the density is renormalised over the range the data could
# take. Each family: (scipy distribution, start values, unpack log-params -> scipy kwargs).
GAP_FAMILIES = {
    "exponential": (stats.expon, lambda x: [np.log(x.mean())],
                    lambda t: {"scale": np.exp(t[0])}),
    "lognorm": (stats.lognorm, lambda x: [np.log(np.log(x).std()), np.log(x).mean()],
                lambda t: {"s": np.exp(t[0]), "scale": np.exp(t[1])}),
    "weibull": (stats.weibull_min, lambda x: [np.log(0.6), np.log(np.median(x))],
                lambda t: {"c": np.exp(t[0]), "scale": np.exp(t[1])}),
}


def fit_truncated(family: str, x: np.ndarray, lower: float, upper: float) -> dict:
    """Maximum likelihood for a family truncated to [lower, upper]."""
    dist, start_values, unpack = GAP_FAMILIES[family]

    def nll(theta):
        kw = unpack(theta)
        mass = dist.cdf(upper, **kw) - dist.cdf(lower, **kw)
        if not np.isfinite(mass) or mass <= 0:
            return np.inf
        return -(dist.logpdf(x, **kw).sum() - len(x) * np.log(mass))

    res = optimize.minimize(nll, start_values(x), method="Nelder-Mead",
                            options={"xatol": 1e-8, "fatol": 1e-8, "maxiter": 5000})
    kw = unpack(res.x)
    frozen = dist(**kw)
    mass = frozen.cdf(upper) - frozen.cdf(lower)
    k = len(res.x)
    return {
        "family": family, "params": kw, "k": k, "loglik": -res.fun, "aic": 2 * k + 2 * res.fun,
        "pdf": lambda v, f=frozen, m=mass: np.where((v >= lower) & (v <= upper), f.pdf(v) / m, 0.0),
        "converged": res.success,
        # median of the model on the range the data can take, comparable to the observed median
        "model_median": frozen.ppf((frozen.cdf(lower) + frozen.cdf(upper)) / 2),
    }


def fit_powerlaw_tail(x: np.ndarray, upper: float) -> dict:
    """Power law p(x) ~ x^-alpha on [x_min, upper], truncated at the burst threshold.

    x_min is the candidate that makes the fitted tail closest to the data (smallest KS
    distance), the Clauset-Shalizi-Newman recipe; alpha by maximum likelihood given x_min.
    """
    def fit_alpha(tail, x_min):
        logs = np.log(tail).sum()

        def nll(a):
            norm = x_min ** (1 - a) - upper ** (1 - a)
            return -(len(tail) * np.log((a - 1) / norm) - a * logs)

        a = optimize.minimize_scalar(nll, bounds=(1.001, 6), method="bounded").x
        return a, -nll(a)

    best = None
    for x_min in np.unique(np.quantile(x, np.linspace(0.05, 0.9, 40)).round()):
        tail = np.sort(x[x >= x_min])
        a, ll = fit_alpha(tail, x_min)
        model_cdf = (x_min ** (1 - a) - tail ** (1 - a)) / (x_min ** (1 - a) - upper ** (1 - a))
        empirical = np.arange(1, len(tail) + 1) / len(tail)
        ks = np.abs(model_cdf - empirical).max()
        if best is None or ks < best["ks"]:
            best = {"x_min": x_min, "alpha": a, "loglik": ll, "ks": ks, "n_tail": len(tail),
                    "aic": 2 * 1 - 2 * ll}
    return best


def fit_gaps(x: np.ndarray, upper: float) -> dict:
    fits = {name: fit_truncated(name, x, 0.0, upper) for name in GAP_FAMILIES}
    tail = fit_powerlaw_tail(x, upper)
    # Same tail, same range: does a (doubly truncated) lognormal beat the power law there?
    tail["lognorm_aic"] = fit_truncated("lognorm", x[x >= tail["x_min"]], tail["x_min"], upper)["aic"]
    return {"fits": fits, "tail": tail, "n": len(x), "median": np.median(x),
            "best": min(fits, key=lambda name: fits[name]["aic"])}


gaps = within_burst_gaps(start, end)
upper = BURST_GAP_MINUTES * 60
gap_fits = {side: fit_gaps(gaps.loc[gaps["side"] == side, "gap"].to_numpy(), upper) for side in SIDES}

for side in SIDES:
    g = gap_fits[side]
    print(f"{side}: {g['n']:,} gaps, observed median {g['median']:.0f} s, best by AIC: {g['best']} "
          f"(its median within the burst range: {g['fits'][g['best']]['model_median']:.0f} s)")
    for name, f in g["fits"].items():
        params = ", ".join(f"{k} = {v:.3g}" for k, v in f["params"].items())
        print(f"   {name:12s} {params:28s} AIC {f['aic']:11,.0f}   dAIC {f['aic'] - g['fits'][g['best']]['aic']:9,.0f}")
    t = g["tail"]
    print(f"   power-law tail: alpha = {t['alpha']:.2f} above x_min = {t['x_min']:.0f} s "
          f"({t['n_tail']:,} gaps, KS {t['ks']:.3f});  tail AIC power law {t['aic']:,.0f} vs lognorm {t['lognorm_aic']:,.0f}")

**What this means.**

- **Exponential is rejected on both sides, overwhelmingly** (ΔAIC ≈ 10,000). Gaps are *not*
  "replies at random moments at one steady rate".
- **Lognormal wins the three-way comparison on both sides**, with almost identical
  parameters: e^μ ≈ 39 s, and σ = 1.88 before against 1.80 after.
- **The power-law tail has α = 1.10 before and 1.21 after**, above roughly 2 minutes
  (153 s before, 111 s after).
- **On that tail, power law and lognormal are a tie:** ΔAIC is 1 to 2, so the data can't tell
  them apart there.

The lognormal is the *best of the three*, not a good description. Its median within the
burst range is 38 s, while the observed median is 27 s. The plot below shows why.

**Mechanism: Barabási (2005), "The origin of bursts and heavy tails in human dynamics".**

- **An exponential gap distribution means a Poisson process:** each reply comes at a random
  moment, independent of the last one.
- **What Barabási found instead,** for email and letters, is a **heavy tail**. People keep a
  mental list of things to do and always pick the most urgent one. Answering the chat is one
  item on that list.
- **When the chat is at the top of the list,** we answer within seconds: the fast core of
  10–40 s gaps, rapid back-and-forth.
- **When something else takes priority** (work, cooking, a meeting), the chat waits until that
  is done. Those waits have no typical length; they spread out with p(x) ∝ x^−α. His
  priority-queue model predicts **α ≈ 1**, and we find **1.1–1.2**.

**Careful phrasing:** lognormal and power law can't be separated on our tail, and the tail
only spans about 1.5 decades (from ~2 minutes to the 60-minute cut). So the claim is
**"heavy-tailed, as Barabási describes"**, not "an exact power law".

**For the hypothesis:** whichever family you pick, before and after are the *same* shape with
the same parameters.

In [ ]:
gap_settings = PlotSettings(
    figsize=(10, 5.5),
    title="",
    xlabel="seconds between two messages inside a burst (log scale)",
    ylabel="share of gaps per log10(seconds)",
)


def gap_label(side: str) -> str:
    g = gap_fits[side]
    p = g["fits"][g["best"]]["params"]
    t = g["tail"]
    return (f"{side} the test ({g['n']:,} gaps)\n"
            f"solid: {g['best']}, e^μ = {p['scale']:.0f} s, σ = {p['s']:.2f}\n"
            f"dashed: tail ∝ x^-{t['alpha']:.2f} above {t['x_min']:.0f} s")


def tail_density(side: str):
    """The power-law tail as a share of ALL gaps, so it sits on the same scale as the bars."""
    g, t = gap_fits[side], gap_fits[side]["tail"]
    a, x_min, share = t["alpha"], t["x_min"], t["n_tail"] / g["n"]
    norm = x_min ** (1 - a) - upper ** (1 - a)
    return lambda v: np.where(v >= x_min, share * (a - 1) * v ** (-a) / norm, np.nan)


# Timestamps are whole seconds, so bin edges sit halfway between whole numbers: every bin
# holds at least one possible value, and the 1-3 s range has no artificial empty bins.
gap_bins = np.unique(np.round(np.geomspace(1, upper, 32))) - 0.5
gap_bins = np.append(gap_bins, upper + 0.5)

host = BeforeAfterDensityPlot(gap_settings)
fig, ax = host.plot(
    samples={side: gaps.loc[gaps["side"] == side, "gap"].to_numpy() for side in SIDES},
    bins=gap_bins,
    colors=SIDE_COLORS,
    densities={side: gap_fits[side]["fits"][gap_fits[side]["best"]]["pdf"] for side in SIDES},
    log_x=True,
    labels={side: gap_label(side) for side in SIDES},
    label_positions={"before": (0.60, 0.97), "after": (0.60, 0.80)},
)
grid = np.geomspace(1, upper, 400)
for side in SIDES:
    ax.plot(grid, tail_density(side)(grid) * grid * np.log(10), color=SIDE_COLORS[side],
            linestyle="--", linewidth=2)
ax.set_title("Inside a conversation, the tempo stayed the same", loc="left", fontsize=12)
fig.text(0.01, -0.04, f"Gaps of 1 s up to the {BURST_GAP_MINUTES}-minute burst threshold; gaps starting "
         "00:00-06:59 left out. Fits truncated at the threshold.",
         fontsize=8.5, color="dimgray")
save(fig, "vis3_gaps_within_bursts")

The two histograms lie on top of each other. On a log x-axis with logarithmic bins, the bar
height is "share of gaps per unit of log10(seconds)", and the curves are converted to that
same scale (density × x × ln 10), so bars and curves can be compared directly.

Read it in two parts:

- **The hump at 10–40 s:** the conversation itself.
- **The long, almost flat shelf from about 2 minutes to the threshold:** the priority-queue
  waits. A shelf that is flat on a log axis is exactly what p(x) ∝ x^−1 looks like, which is
  why the dashed power-law lines follow it and the lognormal (solid) falls away too early.

### Step 3 — the layer that *should* change: fatigue and nausea

**What:** flag every message in the two windows that mentions nausea or fatigue, in French,
Dutch or English. Then count flagged messages per calendar day (zero days included, as in
step 1).

**How the matching works:**

- **Normalise first:** lowercase, straighten apostrophes, remove accents (`écœurée` →
  `ecoeuree`), so one pattern covers every spelling.
- **Word boundaries everywhere**, so that `crevée` matches but `crevettes` (shrimp) doesn't,
  `claquée` but not `claquement`. `plus d'énergie` ("more energy") is not on the list; only the
  negative forms (`pas d'énergie`, `geen energie`, `no energy`) are.
- **These are not the patterns from §4.6.10.** Those (from `roos_03.3`) have no word boundaries
  in the French part, so `crev[ée]e?` also matches `crevettes`. The lists here are the new,
  stricter ones.

The cell **asserts** that none of the named false positives appear in any match, and prints
the most frequent matched terms so the list can be checked by eye.

In [ ]:
def normalise(text: str) -> str:
    """Lowercase, curly apostrophes straightened, accents removed: 'Écœurée' -> 'ecoeuree'."""
    text = str(text).lower().replace("’", "'").replace("œ", "oe")
    return "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))


# Patterns run on normalised text, so no accents in them. \b everywhere: 'crevee' must not
# match 'crevettes', 'claquee' not 'claquement'. 'plus d'energie' (more energy) is not in the
# list; only 'pas d'energie' / 'geen energie' / 'no energy' are.
SYMPTOM_PATTERNS = {
    "nausea": re.compile(
        r"\b(?:"
        r"nausees?|nauseeuses?|nauseeux|(?:envie de )?vomir|ecoeuree?s?|barbouillee?s?|"
        r"misselijk(?:heid)?|overgeven|kotsen|braken|"
        r"nausea|nauseous|vomit(?:ing|ed)?|throw(?:ing)? up|feel(?:ing)? sick|morning sickness"
        r")\b"
    ),
    "fatigue": re.compile(
        r"\b(?:"
        r"fatiguee?s?|crevee?s?|epuisee?s?|naze|claquee?s?|j'?\s?ai sommeil|pas d'energie|"
        r"moe|vermoeid|uitgeput|doodop|slaperig|geen energie|"
        r"tired|exhausted|sleepy|knackered|drained|worn out|no energy"
        r")\b"
    ),
}
EXCLUDED_WORDS = ["lessive", "crevette", "claquement", "plus d'energie"]

in_window = chat[(chat["ts_local"] >= start) & (chat["ts_local"] < end)].copy()
in_window["normalised"] = in_window["message"].map(normalise)
for name, pattern in SYMPTOM_PATTERNS.items():
    in_window[name] = in_window["normalised"].str.contains(pattern)
in_window["either"] = in_window["nausea"] | in_window["fatigue"]

# Guard against the false positives named in the brief: none of these words may be inside a match.
matched_terms = Counter(
    m.group(0) for name, pattern in SYMPTOM_PATTERNS.items()
    for text in in_window.loc[in_window[name], "normalised"] for m in pattern.finditer(text)
)
false_positives = [term for term in matched_terms if any(bad in term for bad in EXCLUDED_WORDS)]
if false_positives:
    raise ValueError(f"excluded words inside a match: {false_positives}")
print(in_window[["nausea", "fatigue", "either"]].sum().rename("messages in the two windows"))
print("\nmost frequent matched terms:", matched_terms.most_common(20))

**Check before trusting it:** every frequent term above is a genuine symptom word, and the
assert passed (no `lessive`, `crevette`, `claquement`, `plus d'énergie`). Two things the
keyword match *can't* separate, worth keeping in mind:

- **Ordinary tiredness vs pregnancy tiredness:** "too tired to correct it" counts the same as
  first-trimester exhaustion. This is why fatigue already had a baseline before the test.
- **Having a symptom vs talking about one:** a pasted "week 6: fatigue, morning nausea…" list
  from a pregnancy app counts too. For *this* hypothesis that is fine, because the claim is
  about what we *talked about*.

⚠️ **The next cell prints short snippets around matches, for checking only. Clear its output
before submitting.**

In [ ]:
# CHECK ONLY -- delete this cell's output before submitting. Short snippets around each match,
# never whole messages.
def snippet(text: str, pattern: re.Pattern, width: int = 30) -> str:
    m = pattern.search(text)
    if m is None:
        return ""
    return "…" + text[max(0, m.start() - width): m.end() + width] + "…"


for name, pattern in SYMPTOM_PATTERNS.items():
    sample = in_window[in_window[name]].sample(8, random_state=1)
    print(f"--- {name}")
    for text in sample["normalised"]:
        print("  ", snippet(text, pattern))

Per-day rates on both sides, plus two shuffle tests (`NullDistribution`, as in 04.4 §4.4.3):
shuffle the before/after label across the 154 days 2,000 times and see how often a random
split produces a difference as large as the real one. One test for the symptom rate, and one
for messages per day from step 1, so both layers get the same test.

In [ ]:
symptom_days = (
    in_window.groupby(in_window["ts_local"].dt.normalize())[["nausea", "fatigue", "either"]].sum()
    .reindex(pd.date_range(start, end - pd.Timedelta(days=1), freq="D"), fill_value=0)
    .rename_axis("day").reset_index()
)
symptom_days["side"] = np.where(symptom_days["day"] < EVENT_DATE, "before", "after")
symptom_rates = symptom_days.groupby("side")[["nausea", "fatigue", "either"]].mean().loc[SIDES]
print(symptom_rates.round(2))

null_symptoms = NullDistribution(
    lambda days: days.groupby("side")["either"].mean().pipe(lambda m: m["after"] - m["before"]),
    n_iter=2000, seed=4,
).run(symptom_days, label="side")
null_daily = NullDistribution(
    lambda days: days.groupby("side")["messages"].mean().pipe(lambda m: m["after"] - m["before"]),
    n_iter=2000, seed=4,
).run(per_day, label="side")
print(f"\nfatigue-or-nausea messages/day, after - before: {null_symptoms.observed:+.2f}, two-sided p = {null_symptoms.p_value():.4f}")
print(f"messages/day, after - before:                   {null_daily.observed:+.1f}, two-sided p = {null_daily.p_value():.4f}")

**What this means.**

- **Fatigue-or-nausea mentions roughly quadrupled:** 0.52 → 2.01 messages a day.
- **Nausea went from zero mentions** in the 77 days before the test to 0.45 a day after.
- **The symptom rise is very unlikely to be chance:** p ≈ 0.001 for the symptom rate, the
  smallest value 2,000 shuffles can resolve, so read it as "p < 0.001".
- **The messages-per-day rise could easily be chance:** p ≈ 0.25 at 11 weeks.

That is the hypothesis's pattern: the content moved, the volume didn't distinguishably move.

One caveat about this test: neighbouring days are not independent (a busy week is busy on
several days), so the shuffle is a little optimistic in both directions.

In [ ]:
bars = (
    symptom_rates.rename(columns={"either": "nausea or fatigue"})
    .reset_index().melt(id_vars="side", var_name="mention", value_name="per_day")
)
symptom_settings = PlotSettings(figsize=(9, 5), title="", xlabel="", ylabel="messages per day that mention it")
fig, ax = BarPlot(symptom_settings).plot(
    data=bars, x="mention", y="per_day", hue="side", hue_order=SIDES, palette=SIDE_COLORS,
    order=["nausea or fatigue", "fatigue", "nausea"], saturation=1,
)
ax.set_xlabel("")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)
ax.legend_.remove()
for side, x_text in zip(SIDES, [0.62, 0.62]):
    ax.text(x_text, 0.92 if side == "before" else 0.82, f"{side} the test (77 days)",
            transform=ax.transAxes, color=SIDE_COLORS[side], fontweight="bold")
before_rate, after_rate = symptom_rates.loc["before", "either"], symptom_rates.loc["after", "either"]
ax.set_title(f"What changed was the content: fatigue and nausea mentions went from "
             f"{before_rate:.2f} to {after_rate:.2f} a day", loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.04, "A message counts once per bar, however many symptom words it contains. "
         "Keyword match in French, Dutch and English.", fontsize=8.5, color="dimgray")
save(fig, "vis3_symptoms_per_day")

This is the one chart where the crimson bar should jump out, and it does. Same colours as
the two rhythm charts, values written on the bars, and a title that states the finding.
`nausea or fatigue` comes first, since it is the measure the result sentence uses.
Fatigue and nausea are shown separately because they behave differently: fatigue has a
before-baseline, nausea starts from zero.

### Step 4 — the parameter table

Every candidate family, both sides, with its parameters, AIC, and n. ΔAIC is measured
against the best family *for that variable and side*. The power-law row is compared only with
the lognormal fitted on the same tail (shown in its parameters column), because it describes a
different subset of the data (the gaps above x_min).

In [ ]:
rows = []
for side in SIDES:
    d = daily[side]
    rows += [
        {"variable": "messages per day", "side": side, "family": "poisson", "parameters": f"μ = {d['poisson_mu']:.1f}",
         "AIC": d["poisson_aic"], "n": d["n_days"]},
        {"variable": "messages per day", "side": side, "family": "nbinom",
         "parameters": f"μ = {d['nb_mu']:.1f}, r = {d['nb_r']:.2f} (α = {d['nb_alpha']:.2f}); variance/mean = {d['vmr']:.1f}",
         "AIC": d["nb_aic"], "n": d["n_days"]},
    ]
    g = gap_fits[side]
    for name, f in g["fits"].items():
        p = f["params"]
        if name == "exponential":
            text = f"mean = {p['scale']:.0f} s"
        elif name == "lognorm":
            text = f"median = {p['scale']:.1f} s, σ = {p['s']:.2f}"
        else:
            text = f"shape = {p['c']:.2f}, scale = {p['scale']:.0f} s"
        rows.append({"variable": "gaps within bursts", "side": side, "family": name + " (truncated)",
                     "parameters": text, "AIC": f["aic"], "n": g["n"]})
    t = g["tail"]
    rows.append({"variable": "gaps within bursts", "side": side, "family": "power-law tail",
                 "parameters": f"α = {t['alpha']:.2f}, x_min = {t['x_min']:.0f} s (lognorm on same tail: AIC {t['lognorm_aic']:,.0f})",
                 "AIC": t["aic"], "n": t["n_tail"]})
param_table = pd.DataFrame(rows)
param_table["ΔAIC"] = param_table["AIC"] - param_table.groupby(["variable", "side", param_table["family"] == "power-law tail"])["AIC"].transform("min")
pd.set_option("display.max_colwidth", 120)
param_table.round({"AIC": 0, "ΔAIC": 0})

### Step 5 — robustness: does the conclusion survive other reasonable choices?

Two choices could have been made differently, so both are varied:

- **The burst threshold:** 15, 30 or 60 minutes.
- **The window length:** 4, 8 or 11 weeks on each side.

For the daily fit, each window also gets the same shuffle test as step 3.

In [ ]:
robust_rows = []
for minutes in [15, 30, 60]:
    g = within_burst_gaps(start, end, minutes)
    fits = {side: fit_gaps(g.loc[g["side"] == side, "gap"].to_numpy(), minutes * 60) for side in SIDES}
    ln = {side: fits[side]["fits"]["lognorm"]["params"] for side in SIDES}
    robust_rows.append({
        "check": f"burst threshold {minutes} min",
        "best family (before / after)": f"{fits['before']['best']} / {fits['after']['best']}",
        "before": f"median {ln['before']['scale']:.0f} s, σ {ln['before']['s']:.2f} (n {fits['before']['n']:,})",
        "after": f"median {ln['after']['scale']:.0f} s, σ {ln['after']['s']:.2f} (n {fits['after']['n']:,})",
        "change": f"median {ln['after']['scale'] / ln['before']['scale'] - 1:+.0%}",
        "shuffle p": "",
    })
for weeks in [4, 8, 11]:
    days = messages_per_day(weeks * 7)
    fits = {side: fit_daily(days.loc[days["side"] == side, "messages"].to_numpy().astype(float)) for side in SIDES}
    best = {side: "nbinom" if fits[side]["nb_aic"] < fits[side]["poisson_aic"] else "poisson" for side in SIDES}
    null = NullDistribution(
        lambda d: d.groupby("side")["messages"].mean().pipe(lambda m: m["after"] - m["before"]),
        n_iter=2000, seed=4,
    ).run(days, label="side")
    robust_rows.append({
        "check": f"window {weeks} weeks each side",
        "best family (before / after)": f"{best['before']} / {best['after']}",
        "before": f"μ {fits['before']['nb_mu']:.0f}, r {fits['before']['nb_r']:.2f} ({weeks * 7} days)",
        "after": f"μ {fits['after']['nb_mu']:.0f}, r {fits['after']['nb_r']:.2f} ({weeks * 7} days)",
        "change": f"μ {fits['after']['nb_mu'] / fits['before']['nb_mu'] - 1:+.0%}",
        "shuffle p": f"{null.p_value():.3f}",
    })
robustness = pd.DataFrame(robust_rows)
robustness

In [ ]:
# Where does the window-dependence come from? Messages per week (Thursday-Wednesday, so the
# last "before" week ends the day before the test), with the number of days under 20 messages.
weekly_volume = (
    per_day.set_index("day")["messages"].resample("W-WED")
    .agg(messages="sum", quiet_days=lambda s: int((s < 20).sum()))
)
weekly_volume.index = weekly_volume.index.date
weekly_volume.rename_axis("week ending").T

**What this means.**

**Gaps within bursts: robust.**

- Lognormal wins on both sides at every threshold.
- The before/after change in the median parameter stays within ±5%.
- The parameters themselves depend on the threshold: a longer cut lets more of the heavy
  tail in, so e^μ goes 28 → 33 → 39 s and σ rises. That is expected for a heavy tail and is
  the same on both sides. (At 30 minutes the numbers match the earlier independent parse:
  e^μ ≈ 33 s, σ 1.71 / 1.64.)
- *The tempo of our conversations did not change* holds however a burst is defined.

**Messages per day: not robust.**

- The family holds (negative binomial everywhere), but the size of the change depends on the
  window: μ +58% at 4 weeks and +37% at 8 weeks (both p ≈ 0.01), against +12% at 11 weeks
  (p ≈ 0.25).
- The weekly table below shows where that comes from: the two weeks ending 14 and 21 August,
  right before the test, are by far the quietest in the window (198 and 246 messages), with
  several near-empty days.
- The shorter the window, the more weight those two weeks get.

**Explained:** we were together in mid-August 2024. Those quiet weeks are exactly the
"together" days from step 1's mechanism (when we are in the same place, the chat goes nearly
silent), not an effect of the test. The 11-week window is the fairer comparison because it
holds that one visit as one of many ups and downs, instead of letting it dominate a short
window. So: *over 11 weeks, daily volume did not change beyond its normal spread; the rise in
shorter windows is the end of a visit, not the pregnancy.*

### What the data can and cannot show

**Remaining confounders, named explicitly:**

- **The end of the summer holidays.** 29 August is roughly when "back to work" starts. A
  return to routine could on its own raise the daily volume (people message more from a desk)
  and the fatigue mentions (work is tiring). The event and the end of summer can't be
  separated with one event in one chat.
- **Time together in mid-August 2024** (confirmed). It explains the window-dependence of the
  daily volume layer, and leaves the gaps and the symptoms untouched.
- **Keyword fatigue is generic.** Part of the fatigue rise could be ordinary tiredness from the
  return to work rather than pregnancy. **Nausea is the cleaner signal:** zero mentions in 11
  weeks before, regular mentions after, and nausea is not something a work week produces.

**Correlation is not causation.**

- **What the data show:** the symptom words rise sharply at the event. The timing fits early
  pregnancy: nausea appears in the first weeks after a positive test.
- **What the data cannot show:** that the pregnancy *caused* the rise (one event, no control
  period with the same calendar), or whether a mention means *having* a symptom or *talking
  about* one.
- **Scope:** n = 2 people, one relationship. Every result describes this chat, not
  pregnancies in general.

### The result, as a sentence with parameters in it

Generated from the fitted values above, so every number comes from a cell output. It names the
mid-August visit, because that is what makes shorter windows show a rise in messages per day
(step 5).

In [ ]:
d_b, d_a = daily["before"], daily["after"]
g_b, g_a = (gap_fits[s]["fits"]["lognorm"]["params"] for s in SIDES)
result_sentence = (
    f"Before the positive test we sent on average μ = {d_b['nb_mu']:.0f} messages a day "
    f"(negative binomial, r = {d_b['nb_r']:.1f}), afterwards μ = {d_a['nb_mu']:.0f} (r = {d_a['nb_r']:.1f}); "
    f"a difference a random split of the same days gives often (p = {null_daily.p_value():.2f}); "
    f"shorter windows show a rise only because we were together, and quiet, in mid-August. "
    f"Gaps within bursts stayed lognormal with median parameter e^μ = {g_b['scale']:.0f} s → {g_a['scale']:.0f} s "
    f"(σ = {g_b['s']:.2f} before, {g_a['s']:.2f} after). "
    f"The tempo of our conversations stayed the same; the content changed: fatigue and nausea "
    f"mentions went from {before_rate:.2f} to {after_rate:.2f} a day."
)
print(result_sentence)

## 4.6.13 The hard weeks: does weeks 5–15 of the pregnancy stand out?

**My story: "How to prepare for a next pregnancy."** The difficult part of the pregnancy took
about ten weeks, from week 5 to week 15. This section checks whether the chat confirms that
memory: does that period stand out against the rest of our conversation, how strongly, and
which weeks were the worst?

**What the chat can and cannot tell.** It can show *when* the hard period started and ended,
*how intense* it was compared with normal times, and *which* symptoms dominated. It cannot
show what helped or what we should do differently; those lessons are mine to draw from it.

**Periods.** Pregnancy weeks are counted from `PREGNANCY_START` (§4.6.10: Benji's birth − 41
weeks = 28 July 2024), with the same convention `roos_03.3` used: week *n* starts at
`PREGNANCY_START` + *n* weeks.

| period | dates | why |
|---|---|---|
| before pregnancy (long-distance) | chat start – 28 Aug 2024 | the baseline: same channel, we lived apart |
| test to week 5 | 29 – 31 Aug 2024 | 3 days, left out of the comparison |
| **hard weeks 5–15 (long-distance)** | **1 Sep – 9 Nov 2024** | the period in question, 10 weeks |
| rest of pregnancy | 10 Nov 2024 – 10 May 2025 | from here on we lived together |
| after Benji | 11 May 2025 – end of the export | |

Choices flagged so I can overrule them:

- **"Until week 15"** is read as *up to the start of week 15*, which gives exactly the 10 weeks
  I remember.
- **The baseline includes weeks 0–4 of the pregnancy**, before we knew. If the symptoms
  already started then, that makes the baseline slightly higher, so the comparison below is
  conservative rather than flattering.
- **The main comparison is hard weeks vs. before**, because both are long-distance: anything
  we felt, we had to *write*. From the move onwards most of this was said in person, so the
  later periods are shown but are lower bounds.
- **Rates per 1,000 messages**, not only per day: after the move we wrote half as much, and a
  per-day rate would mostly measure that.

In [ ]:
# Pregnancy weeks counted from PREGNANCY_START (§4.6.10: BABY_DATE - 41 weeks), with the same
# convention roos_03.3 used: week n starts at PREGNANCY_START + n weeks.
HARD_START = PREGNANCY_START + pd.Timedelta(weeks=5)  # start of week 5
HARD_END = PREGNANCY_START + pd.Timedelta(weeks=15)  # end of week 14 = 10 weeks, exclusive

PERIODS = {  # label: [start, end)
    "before pregnancy (long-distance)": (chat["ts_local"].min().normalize(), EVENT_DATE),
    "test to week 5": (EVENT_DATE, HARD_START),
    "hard weeks 5-15 (long-distance)": (HARD_START, HARD_END),
    "rest of pregnancy": (HARD_END, BABY_DATE),
    "after Benji": (BABY_DATE, chat["ts_local"].max().normalize() + pd.Timedelta(days=1)),
}
PERIOD_ORDER = [p for p in PERIODS if p != "test to week 5"]  # the 3 days in between are left out
HARD = "hard weeks 5-15 (long-distance)"
BEFORE = "before pregnancy (long-distance)"


def period_of(ts: pd.Series) -> pd.Series:
    labels = pd.Series(pd.NA, index=ts.index, dtype="object")
    for name, (lo, hi) in PERIODS.items():
        labels[(ts >= lo) & (ts < hi)] = name
    return labels


# Every message in the chat, flagged with the symptom patterns from step 3 of §4.6.12.
whole = chat.copy()
whole["normalised"] = whole["message"].map(normalise)
for name, pattern in SYMPTOM_PATTERNS.items():
    whole[name] = whole["normalised"].str.contains(pattern)
whole["either"] = whole["nausea"] | whole["fatigue"]
whole["period"] = period_of(whole["ts_local"])

for name, (lo, hi) in PERIODS.items():
    days = (hi - lo).days
    n = int((whole["period"] == name).sum())
    print(f"{name:34s} {lo.date()} to {(hi - pd.Timedelta(days=1)).date()}   {days:4d} days   "
          f"{n:6,} messages   {n / days:5.1f}/day")

During the hard weeks the chat was also *busier* than in the long-distance year before
(96.6 against 83.9 messages a day). From the move onwards, volume drops to about 43 and then
34 a day: the in-person effect in one number, and the reason the comparisons below use a rate
per 1,000 messages.

In [ ]:
# One row per calendar day, zero days included, so "per day" means per calendar day.
days_all = (
    whole.groupby(whole["ts_local"].dt.normalize())[["nausea", "fatigue", "either"]].sum()
    .join(whole.groupby(whole["ts_local"].dt.normalize()).size().rename("messages"))
)
days_all = days_all.reindex(
    pd.date_range(days_all.index.min(), days_all.index.max(), freq="D"), fill_value=0
).rename_axis("day").reset_index()
days_all["period"] = period_of(days_all["day"])

totals = days_all.groupby("period")[["messages", "nausea", "fatigue", "either"]].sum().loc[PERIOD_ORDER]
n_days = days_all.groupby("period").size().loc[PERIOD_ORDER]
period_table = pd.DataFrame({
    "days": n_days,
    "messages/day": totals["messages"] / n_days,
    "nausea/day": totals["nausea"] / n_days,
    "fatigue/day": totals["fatigue"] / n_days,
    "either/day": totals["either"] / n_days,
    # per 1,000 messages: corrects for how much we wrote at all in that period
    "nausea per 1,000 msgs": 1000 * totals["nausea"] / totals["messages"],
    "fatigue per 1,000 msgs": 1000 * totals["fatigue"] / totals["messages"],
    "either per 1,000 msgs": 1000 * totals["either"] / totals["messages"],
    "symptom messages": totals["either"],
})
period_table.round(2)

**What this means.**

- **Nausea is almost exclusive to the hard weeks:** 5.2 mentions per 1,000 messages, against
  0.1–0.4 in every other period.
- **Fatigue doubled:** 17.9 per 1,000 against 8.2 before. Fatigue has a real baseline (ordinary
  tiredness counts too), so the doubling is the signal, not the level.
- **Either symptom:** 22.6 per 1,000 in the hard weeks. That is 2.7 times the long-distance
  baseline, and well above both later periods (7.3 and 10.0).
- **After Benji, fatigue (9.7 per 1,000) is above the before-pregnancy baseline**, even though
  most tiredness was by then talked about in person. That number is a lower bound.

In [ ]:
# Same shuffle test as §4.6.12, on the two long-distance periods only: the hard weeks against
# the whole long-distance time before the pregnancy.
pair = days_all[days_all["period"].isin([BEFORE, HARD])].copy()


def share_difference(frame: pd.DataFrame) -> float:
    """Symptom messages per 1,000 messages, hard weeks minus before."""
    sums = frame.groupby("period")[["either", "messages"]].sum()
    share = 1000 * sums["either"] / sums["messages"]
    return share[HARD] - share[BEFORE]


hard_null = NullDistribution(share_difference, n_iter=2000, seed=4).run(pair, label="period")
ratio = period_table.loc[HARD, "either per 1,000 msgs"] / period_table.loc[BEFORE, "either per 1,000 msgs"]
print(f"symptom messages per 1,000: hard weeks minus before = {hard_null.observed:+.1f} "
      f"(x{ratio:.1f}), two-sided p = {hard_null.p_value():.4f}")

A random split of the same days almost never produces a difference this large (p ≈ 0.001,
the smallest value 2,000 shuffles can resolve). As in §4.6.12, neighbouring days are not
fully independent, so read it as "very unlikely to be chance", not as an exact number.

### Week by week: would a hard week be unusual in normal times?

A period average can hide one extreme week. So each of the 10 hard weeks is judged on its
own, the lesson-4 way (§4.5.5, "a point in the tail is a question, not a verdict"):

1. **Fit a family to the normal weeks.** Every full week of the long-distance time before
   the test. Weekly symptom counts are counts, and how much we talk about being tired
   wanders from week to week, so the negative binomial is the candidate again (same
   gamma-Poisson mechanism as messages per day), checked against Poisson on AIC.
2. **Score each hard week by its tail probability under that family:** how often would a
   normal week produce *at least* this many symptom messages? This uses the family we
   committed to, not a z-score that silently assumes a normal distribution.

Weekly counts are compared directly because the hard weeks and the normal weeks have similar
volume (96.6 and 83.9 messages a day). The last column is a volume check anyway: the share
per 1,000 messages, as a percentile among the normal weeks.

In [ ]:
# One row per pregnancy-week-sized block over the whole chat (week 0 = PREGNANCY_START), so
# the hard period is exactly weeks 5-14 and every block is the same 7 weekdays.
days_all["week"] = (days_all["day"] - PREGNANCY_START).dt.days // 7
weekly = days_all.groupby("week").agg(
    start=("day", "min"), n_days=("day", "size"),
    messages=("messages", "sum"), either=("either", "sum"),
    nausea=("nausea", "sum"), fatigue=("fatigue", "sum"),
)
weekly = weekly[weekly["n_days"] == 7].copy()  # drop the partial first and last block
weekly["share"] = 1000 * weekly["either"] / weekly["messages"].where(weekly["messages"] > 0)
weekly["group"] = np.select(
    [weekly.index < 4, weekly.index.isin(range(5, 15))],
    ["before pregnancy", "hard weeks"], default="other",
)

baseline = weekly.loc[weekly["group"] == "before pregnancy", "either"].to_numpy().astype(float)
baseline_fit = fit_daily(baseline)  # same Poisson-vs-nbinom MLE as §4.6.12, on weekly counts
nb_week = baseline_fit["nb_dist"]
print(f"normal long-distance weeks: {len(baseline)} weeks, nbinom μ = {baseline_fit['nb_mu']:.1f} "
      f"symptom messages/week, r = {baseline_fit['nb_r']:.2f}  "
      f"(AIC nbinom {baseline_fit['nb_aic']:.0f} vs poisson {baseline_fit['poisson_aic']:.0f})")

hard_weeks = weekly[weekly["group"] == "hard weeks"].copy()
# Tail probability under the family fitted to normal weeks: how often would an ordinary
# long-distance week produce at least this many symptom messages? (§4.5.5: a point in the tail
# is judged against the family you committed to, not a z-score.)
hard_weeks["P(normal week ≥ this)"] = nb_week.sf(hard_weeks["either"] - 1)
hard_weeks["share percentile vs normal weeks"] = hard_weeks["share"].map(
    lambda s: (weekly.loc[weekly["group"] == "before pregnancy", "share"] < s).mean()
)
cutoff = nb_week.ppf(0.99)
print(f"99th percentile of a normal week: {cutoff:.0f} symptom messages; "
      f"hard weeks above it: {(hard_weeks['either'] > cutoff).sum()} of {len(hard_weeks)}")
hard_weeks["start"] = hard_weeks["start"].dt.date
hard_weeks.rename_axis("pregnancy week")[
    ["start", "messages", "either", "nausea", "fatigue", "P(normal week ≥ this)", "share percentile vs normal weeks"]
].round(4)

**What this means.**

- **The normal weeks fit a negative binomial** (μ = 4.9 symptom messages a week, r = 5.26),
  preferred over Poisson by ΔAIC ≈ 10.
- **6 of the 10 hard weeks are beyond the 99th percentile** of a normal week (more than 14
  symptom messages). The worst, **week 12 (from 20 October): 30 symptom messages**, which a
  normal week would essentially never produce.
- **The edges of the period are its mildest weeks:** week 5 (10) and week 14 (8). The chat
  confirms my memory of when it started and stopped.
- **Nausea came in waves:** none in weeks 9 and 11, peaks of 6–7 in weeks 6, 12 and 13.
  Fatigue was there every week (7 to 25 messages).
- **Volume doesn't explain it:** measured as a share of that week's messages, every hard week
  sits above at least 88% of the normal weeks.

In [ ]:
timeline_settings = PlotSettings(
    figsize=(12, 5),
    title="",
    xlabel="",
    ylabel="messages mentioning nausea or fatigue\nper 1,000 messages (weekly)",
)
host = LinePlot(timeline_settings)
fig, ax = host.plot(data=weekly, x="start", y="share", gaps=True, color="#7f7f7f", linewidth=1.2)
ax.axvspan(HARD_START, HARD_END, color=timeline_settings.highlight_color, alpha=0.15, lw=0)
hard_line = weekly[weekly["group"] == "hard weeks"]
ax.plot(hard_line["start"], hard_line["share"], color=timeline_settings.highlight_color, linewidth=2.5)
ax.text(HARD_START + (HARD_END - HARD_START) / 2, 0.97, "weeks 5-15", transform=ax.get_xaxis_transform(),
        ha="center", va="top", color=timeline_settings.highlight_color, fontweight="bold")
# MOVE_DATE is the notebook's approximate move date (§4.6.4), hence "around".
for date, label, height in [(str(MOVE_DATE.date()), "moved in together (around)", 0.68),
                            (str(BABY_DATE.date()), "Benji is born", 0.88)]:
    host.plot_on(EventMarker(timeline_settings), date=date, label=label, height=height, text_offset_days=20)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(
    f"Weeks 5-15 stand out from the long-distance year before: {period_table.loc[HARD, 'either per 1,000 msgs']:.0f} "
    f"symptom messages per 1,000, against {period_table.loc[BEFORE, 'either per 1,000 msgs']:.0f}",
    loc="left", fontsize=12,
)
fig.text(0.01, -0.03, "After the move, tiredness was mostly talked about in person, so the chat undercounts it there; "
         "after Benji, weeks hold few messages, so one mention moves the line a lot.", fontsize=8.5, color="dimgray")
save(fig, "trimester_timeline")

The crimson stretch is the hard weeks; grey is everything else, the same grey-first rule as
§4.6.12.

- **Against the long-distance year before**, there is no comparable *sustained* run.
  Single weeks come close to the hard weeks' level, but never ten weeks in a row.
- **After Benji, single weeks spike higher than any hard week.** Those weeks hold few messages
  (about 34 a day), so one or two mentions move the line a lot, and the period as a whole
  stays at 10 per 1,000. That is noise from a small denominator, not a second hard period.
- **The "moved in together" marker is the notebook's approximate `MOVE_DATE`** (1 November
  2024, from §4.6.4). It falls inside the last weeks of the hard period, while I remember the
  move as mid-November. If the real date is later, the whole hard period was long-distance.

### Testing the stand-out chart before trusting it

Four checks, one per claim the next chart makes:

1. **Is the negative binomial a fair description of the normal weeks?** A parametric
   bootstrap. Draw 54 weeks from the fitted family, refit, and measure the largest gap between
   the data's cumulative curve and the fit's, 1,000 times. If the real data sit no further
   from their fit than these simulated samples do, the family is plausible.
2. **Does the 99% line hold in the real data**, not only in the fitted family?
3. **How surprising is "6 of 10"?** If the hard weeks were ordinary weeks, each would cross
   the line with the small probability from (2); the binomial gives the chance of 6 or more
   out of 10. This treats the weeks as independent, which neighbouring weeks are not quite,
   so read the result as an order of magnitude.
4. **Why the 10 hard weeks get no family of their own.** Fit one anyway, then resample the
   10 weeks 1,000 times and see how firmly the parameters are pinned down.

In [ ]:
rng = np.random.default_rng(4)
N_BOOT = 1000

# 1. Does the negative binomial actually describe the normal weeks? Parametric bootstrap:
#    draw 54 weeks from the fitted family, refit, and see how far the refitted family sits
#    from its own sample (largest gap between the two cumulative curves). If the real data
#    sit no further from their fit than these simulated samples do, the family is plausible.
def cdf_gap(sample: np.ndarray, dist) -> float:
    grid = np.arange(0, sample.max() + 1)
    empirical = np.searchsorted(np.sort(sample), grid, side="right") / len(sample)
    return float(np.abs(empirical - dist.cdf(grid)).max())


observed_gap = cdf_gap(baseline, nb_week)
boot_gaps = []
for _ in range(N_BOOT):
    sim = nb_week.rvs(len(baseline), random_state=rng).astype(float)
    boot_gaps.append(cdf_gap(sim, fit_daily(sim)["nb_dist"]))
gof_p = (np.array(boot_gaps) >= observed_gap).mean()
print(f"1. fit check, normal weeks: largest cdf gap {observed_gap:.3f}; "
      f"simulated samples from the family have a gap this large in {gof_p:.0%} of cases")

# 2. Does the 99% line hold in the real normal weeks, not only in the fitted family?
p_beyond = nb_week.sf(cutoff)  # P(normal week > cutoff) under the family
print(f"2. normal weeks above the 99% line ({cutoff:.0f}): {(baseline > cutoff).sum()} of {len(baseline)} "
      f"(the family expects {p_beyond * len(baseline):.1f})")

# 3. How likely are 6 or more out of 10 such weeks, if the hard weeks were normal weeks?
#    Treats the 10 weeks as independent, which neighbouring weeks are not quite.
n_above = int((hard_weeks["either"] > cutoff).sum())
p_six = stats.binom.sf(n_above - 1, len(hard_weeks), p_beyond)
print(f"3. chance that 10 normal weeks contain {n_above} or more beyond the line: {p_six:.1e}")

# 4. Why no family on the 10 hard weeks: fit one anyway, and resample the 10 weeks to see
#    how well its parameters are pinned down.
hard_counts = hard_weeks["either"].to_numpy().astype(float)
hard_fit = fit_daily(hard_counts)
boot_mu, boot_r = [], []
for _ in range(N_BOOT):
    resample = rng.choice(hard_counts, len(hard_counts), replace=True)
    f = fit_daily(resample)
    boot_mu.append(f["nb_mu"])
    boot_r.append(f["nb_r"])
lo_mu, hi_mu = np.percentile(boot_mu, [5, 95])
lo_r, hi_r = np.percentile(boot_r, [5, 95])
r_range = f"{lo_r:.1f}-{hi_r:.1f}" if hi_r < 1000 else f"{lo_r:.1f} to no upper limit (= Poisson)"
print(f"4. nbinom on the 10 hard weeks: μ = {hard_fit['nb_mu']:.1f} (90% range {lo_mu:.1f}-{hi_mu:.1f}), "
      f"r = {hard_fit['nb_r']:.1f} (90% range {r_range});  "
      f"AIC nbinom {hard_fit['nb_aic']:.1f} vs poisson {hard_fit['poisson_aic']:.1f}")

**What this means.**

1. **The family fits.** Simulated samples sit this far from their own fit in most cases, so
   the normal weeks are what a negative binomial produces.
2. **The line holds:** none of the 54 real normal weeks crosses it.
3. **6 of 10 by chance is practically impossible**, even allowing for the independence
   caveat. This is the test of the chart's claim.
4. **The 10 hard weeks are too few for a family:** a negative binomial and a Poisson fit
   them almost equally well (ΔAIC ≈ 1), and r can't be pinned down (anything from about 7 to
   no limit). What *is* pinned down is their average: about 15 symptom messages a week,
   against 4.9 in normal weeks. So the footnote under the chart is correct, and it says
   precisely what is too little: the data are enough for an average, not for a shape.

In [ ]:
standout_settings = PlotSettings(
    figsize=(10, 5.5),
    title="",
    xlabel="messages mentioning nausea or fatigue, per week",
    ylabel="share of normal weeks",
)
top = int(max(weekly.loc[weekly["group"] != "other", "either"].max(), baseline.max())) + 2
fig, ax = BeforeAfterDensityPlot(standout_settings).plot(
    samples={"before": baseline},
    bins=np.arange(-0.5, top + 1, 1),
    colors=SIDE_COLORS,
    densities={"before": nb_week.pmf},
    discrete=True,
    labels={"before": f"{len(baseline)} normal long-distance weeks, fitted negative binomial:\n"
                      f"average μ = {baseline_fit['nb_mu']:.1f} symptom messages a week\n"
                      f"r = {baseline_fit['nb_r']:.2f}: how much that average wanders\n"
                      f"from week to week (smaller r = more)"},
    label_positions={"before": (0.42, 0.97)},
)
ax.set_xlim(-0.5, top + 0.5)
ymax = ax.get_ylim()[1]
# One dot per hard week, stacked when two weeks share a count, so all 10 stay visible.
stack = hard_weeks.groupby("either").cumcount()
ax.scatter(hard_weeks["either"], ymax * (0.02 + 0.035 * stack), color=SIDE_COLORS["after"], s=60, zorder=3)
ax.axvline(cutoff, color="dimgray", linestyle=":", linewidth=1)
ax.text(cutoff, ymax * 0.35, " 99% of normal\n weeks below", color="dimgray", fontsize=9, va="center")
ax.text(0.42, 0.62, f"the 10 hard weeks (dots): {n_above} of 10\nbeyond what a normal week produces",
        transform=ax.transAxes, color=SIDE_COLORS["after"], fontweight="bold", va="top")
ax.set_title(f"{n_above} of the 10 hard weeks would be a 1-in-100 week in normal times",
             loc="left", fontsize=12, color="dimgray")
fig.suptitle("How to survive a pregnancy: prepare and plan.", x=0.125, ha="left",
             fontsize=15, fontweight="bold")
fig.text(0.01, -0.05,
         f"No family fitted to the 10 hard weeks: too few to tell a negative binomial from a Poisson. "
         f"Their average is clear: {hard_fit['nb_mu']:.1f} a week (90% range {lo_mu:.1f}-{hi_mu:.1f}).",
         fontsize=8.5, color="dimgray")
save(fig, "trimester_standout")

Grey bars: the 54 normal long-distance weeks, with the fitted negative binomial (μ = the average per week, r = how much that average wanders: a small r means very different weeks, a large r means a steady rate, and r → ∞ is a Poisson). Crimson dots:
the 10 hard weeks, stacked where two weeks share a count. Everything right of the dotted line
is something a normal week produces less than once in a hundred. This is the chart that
answers "does it stand out?" with the lesson-4 tools.

### The same weeks as two probability distributions

The chart above *is* a distribution: with `discrete=True` and one bin per count, each grey bar
is the share of the 54 normal weeks with exactly that many symptom messages, so the bars add up
to 1 and read as a probability, P(week = k). The fitted line is the negative binomial's pmf on
the same scale. Only the crimson dots are not: their height is just stacking, not a probability.

This version puts both groups on the probability scale: grey = the 54 normal weeks with their
fitted family, crimson = the 10 hard weeks (each week is 1/10 = 0.1, two weeks with the same
count make 0.2). No family on the crimson bars, for the reason in check 4 above.

In [ ]:
pmf_settings = PlotSettings(
    figsize=(10, 5.5),
    title="",
    xlabel="messages mentioning nausea or fatigue, per week",
    ylabel="probability: share of weeks with exactly this many",
)
fig, ax = BeforeAfterDensityPlot(pmf_settings).plot(
    samples={"normal": baseline, "hard": hard_counts},
    bins=np.arange(-0.5, top + 1, 1),
    colors={"normal": SIDE_COLORS["before"], "hard": SIDE_COLORS["after"]},
    densities={"normal": nb_week.pmf},
    discrete=True,
    labels={
        "normal": f"{len(baseline)} normal long-distance weeks\n"
                  f"line: fitted negative binomial (μ = {baseline_fit['nb_mu']:.1f}, r = {baseline_fit['nb_r']:.2f})",
        "hard": f"the {len(hard_counts)} hard weeks (no family fitted)\n"
                f"average {hard_counts.mean():.1f} a week",
    },
    label_positions={"normal": (0.58, 0.97), "hard": (0.58, 0.72)},
)
ax.set_xlim(-0.5, top + 0.5)
ax.axvline(cutoff, color="dimgray", linestyle=":", linewidth=1)
ax.text(cutoff, ax.get_ylim()[1] * 0.6, "99% of normal \nweeks below ", color="dimgray", fontsize=9,
        va="center", ha="right")
ax.set_title(f"Normal weeks peak at 2-8 symptom messages; {n_above} of the 10 hard weeks sit beyond {cutoff:.0f}",
             loc="left", fontsize=12, color="dimgray")
print(f"bars add up to: normal {np.histogram(baseline, bins=np.arange(-0.5, top + 1))[0].sum() / len(baseline):.2f}, "
      f"hard {np.histogram(hard_counts, bins=np.arange(-0.5, top + 1))[0].sum() / len(hard_counts):.2f}")
save(fig, "trimester_standout_pmf")

### Before, during and after: the ten weeks against both neighbours

The chart above compares the hard weeks with *normal times* only. This one adds the other
side: the weeks right after the hard period, until Benji's birth (pregnancy weeks 15–40; the
week he was born in is left out). Three columns:

- **Before the pregnancy** (long-distance, the same 54 weeks as above).
- **The hard weeks 5–15** (long-distance).
- **After them** (from about week 15 we lived together).

**Choices:**

- **One dot per week, nothing averaged away.** With only 10 hard weeks, a box or a single
  bar would hide how many of them actually stand out.
- **Per 1,000 messages:** after the move we wrote about half as much, so raw counts would
  mostly show that.
- **The black line is the period pooled** (all its symptom messages ÷ all its messages), the
  same rate as in the period table above.
- **Two shuffle tests**, hard weeks vs. before and hard weeks vs. after, at week level:
  shuffle which group a week belongs to, 2,000 times.

In [ ]:
# Three groups of full weeks: before (long-distance), the hard weeks, and the weeks right
# after them until Benji's birth (living together). Rate per 1,000 messages per week,
# because after the move we wrote about half as much.
baby_week = (BABY_DATE - PREGNANCY_START).days // 7  # the week Benji was born in, left out
weekly["compare"] = np.select(
    [weekly.index < 4, weekly.index.isin(range(5, 15)), weekly.index.isin(range(15, baby_week))],
    ["before", "hard weeks", "after"], default="",
)
compare = weekly[weekly["compare"] != ""].copy()
COMPARE_ORDER = ["before", "hard weeks", "after"]


def pooled_share(frame: pd.DataFrame) -> pd.Series:
    """Symptom messages per 1,000 messages, all weeks of a group pooled."""
    sums = frame.groupby("compare")[["either", "messages"]].sum()
    return 1000 * sums["either"] / sums["messages"]


rates = pooled_share(compare)
tests = {}
for other in ["before", "after"]:
    pair_weeks = compare[compare["compare"].isin(["hard weeks", other])]
    tests[other] = NullDistribution(
        lambda f, other=other: pooled_share(f).pipe(lambda s: s["hard weeks"] - s[other]),
        n_iter=2000, seed=4,
    ).run(pair_weeks, label="compare")
n_weeks = compare["compare"].value_counts()
for group in COMPARE_ORDER:
    print(f"{group:10s} {n_weeks[group]:3d} weeks   {rates[group]:5.1f} per 1,000 messages   "
          f"weeks above the hard-weeks' lowest week: {(compare.loc[compare['compare'] == group, 'share'] > compare.loc[compare['compare'] == 'hard weeks', 'share'].min()).sum()}")
for other, result in tests.items():
    print(f"hard weeks vs {other}: {result.observed:+.1f} per 1,000, shuffle p = {result.p_value():.4f}")

compare_settings = PlotSettings(
    figsize=(10, 5.5), title="", xlabel="",
    ylabel="messages mentioning nausea or fatigue\nper 1,000 messages (one dot = one week)",
    base_color="#7f7f7f",
)
fig, ax = DotStripPlot(compare_settings).plot(
    data=compare, x="compare", y="share", order=COMPARE_ORDER,
    highlight_groups=["hard weeks"],
    summaries=rates.to_dict(),
    summary_labels={g: f"{rates[g]:.1f}" for g in COMPARE_ORDER},
    tick_labels={
        "before": f"before the pregnancy\nlong-distance ({n_weeks['before']} weeks)",
        "hard weeks": f"weeks 5-15\nlong-distance ({n_weeks['hard weeks']} weeks)",
        "after": f"weeks 15-41, until Benji\ntogether ({n_weeks['after']} weeks)",
    },
)
ax.set_title(
    f"The ten hard weeks stand out on both sides: {rates['hard weeks']:.0f} symptom messages per 1,000, "
    f"against {rates['before']:.0f} before and {rates['after']:.0f} after",
    loc="left", fontsize=12,
)
fig.text(0.01, -0.06, "Black line = all weeks of that period pooled. From about week 15 we lived together, so tiredness "
         "was mostly said in person: the right column is a lower bound.", fontsize=8.5, color="dimgray")
save(fig, "trimester_before_during_after")

**What this means.**

- **The hard weeks stand out on both sides:** 22.6 symptom messages per 1,000, against 8.2
  before and 7.3 after.
- **Both differences are very unlikely to be chance** (p ≈ 0.001 each, the smallest value
  2,000 shuffles can resolve).
- **The overlap is small.** Only 6 of the 54 weeks before, and 1 of the 26 weeks after, come
  above the *mildest* hard week. The rest of the hard weeks sit above every other week of the
  pregnancy.

**One honest caveat about the right-hand column:** from about week 15 we lived together, and
most of the tiredness was then said in person, not written. So "after" is a lower bound, and
part of the drop could be the move rather than the symptoms easing. The **left-hand
comparison is the clean one** (same channel, both long-distance). The right-hand one shows
the chat never again reached that level during the pregnancy, but it can't say by how much
the symptoms themselves eased.

In [ ]:
plot_rows = period_table.reset_index().rename(columns={"period": "period"})
plot_rows["label"] = plot_rows["period"].str.replace(" (long-distance)", "\n(long-distance)", regex=False)
period_settings = PlotSettings(figsize=(10, 5), title="", xlabel="",
                               ylabel="symptom messages per 1,000 messages")
palette = {lab: (period_settings.highlight_color if per == HARD else "#7f7f7f")
           for lab, per in zip(plot_rows["label"], plot_rows["period"])}
fig, ax = BarPlot(period_settings).plot(
    data=plot_rows, x="label", y="either per 1,000 msgs", hue="label", palette=palette,
    saturation=1, legend=False,
)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3)
ax.set_xlabel("")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("The hard weeks are the peak of the whole chat", loc="left", fontsize=12)
fig.text(0.01, -0.06, "Nausea or fatigue mentions per 1,000 messages. The last two periods are lower bounds: "
         "living together, most of this was said in person, not in the chat.", fontsize=8.5, color="dimgray")
save(fig, "trimester_periods")

### What this says for "how to prepare for a next pregnancy"

**What the data show**, with numbers from the cells above:

- **The hard period is real and has clear edges.** About 10 weeks of symptom talk at 2.7 times
  the normal rate. It started right at the positive test (week 5) and faded by week 14.
- **It peaked late, around week 12**, not at the start.
- **Nausea came and went; fatigue stayed.** Nausea had weeks with zero mentions; fatigue was
  there every single week.
- **We wrote more, not less**, during those weeks (96.6 against 83.9 messages a day), while we
  still lived apart.

**What the data cannot show:**

- **What helped**, or what would have made it easier.
- **How the later periods compare fairly.** We talked in person by then, so those bars are
  lower bounds.
- **Whether a next pregnancy will follow the same pattern.** This is n = 1 pregnancy.
- **Ordinary tiredness vs. pregnancy tiredness.** The fatigue keywords can't tell them apart;
  nausea is the cleaner signal.

**My lessons for a next pregnancy:** *(write them here, in my own words, from the points
above)*

### Reflectie-tekst

*(To be written once the fifth rubric criterion is confirmed from the course page: it
should refer by name to preattentive processing, text integration, storyline (verhaallijn),
spurious correlations, and that fifth criterion.)*